# Level 1 Project Starter: Adaptive Navigation Agent

Task:

> Find and sort cubes from the source area into their matching destination pads.

Level 1 forbids scene_state and exact entity IDs. Coordinate go_to is allowed only with coordinates estimated or recorded by your system.

Set both `MENLO_API_KEY` and `TOKAMAK_API_KEY` before running. The project requires a text LLM decision loop.


## How To Use This Starter

This is a minimal scaffold, not a solution. SUPPORT CODE gives you setup, wrappers, schema validation, and memory structures. STUDENT TODO sections are where your team designs the LLM decision, observation, action execution, verification, and memory update strategy.


In [ ]:
# Colab/local setup. Run this once if needed.
%pip install -q "git+https://github.com/asimovinc/hansung-menlo-robotics-workshop.git" python-dotenv opencv-python Pillow matplotlib


In [ ]:
# Optional LLM model selection. Edit this before running the starter scaffold.
import os

# 본선 설정: LLM/VLM 모두 qwen/qwen3.6-35b-a3b 사용.
os.environ.setdefault("MENLO_LLM_MODEL", "qwen/qwen3.6-35b-a3b")
os.environ.setdefault("MENLO_VLM_MODEL", "qwen/qwen3.6-35b-a3b")
# Other approved choice (필요 시 주석 해제):
# os.environ["MENLO_LLM_MODEL"] = "minimaxai/minimax-m3"

print("MENLO_LLM_MODEL =", os.environ["MENLO_LLM_MODEL"])
print("MENLO_VLM_MODEL =", os.environ["MENLO_VLM_MODEL"])


## Starter Scaffold

Run this cell, then edit the TODO sections as your design evolves.


## Team Strategy: Sensing-First Adaptive Navigation

본 팀의 Level 1 에이전트는 **오직 관찰(센싱)로 추정/기록한 좌표만으로** 동작한다.
`scene_state`와 정확한 entity ID는 규정상 사용하지 않으며, `go_to` 좌표는 모두 카메라·VLM 관측으로 얻은 값이다. 전체 루프는 4단계 전략으로 구성한다.

### 1) 랜덤 시작 위치 → 360° 센싱 스캔
라운드마다 로봇은 임의 위치에서 시작한다. 첫 cycle에서 제자리 360° 본체 회전 스캔(`_scan360_cube_candidates`)으로 주변 큐브·패드 표지·A(source) 방향을 동시에 관측한다. 큐브가 보이지 않으면 가까운 미탐색 웨이포인트(또는 관측된 A 방향)로 이동 후 재스캔한다 — "생각 없이 즉시 이동" 원칙.

### 2) 큐브 pick 후 목적지 탐색 + 프롬프트 엔지니어링
가장 가까운 큐브를 pick한 뒤, 들고 있는 색에 대응하는 목적지 패드를 찾는다. 목적지 패드는 **고정 표지판 규칙(B red / C green / D blue / E yellow)**과 VLM 표지판 인식으로 확인한다. VLM 프롬프트는 A/conveyor 표지와의 혼동을 차단하도록 설계(`SIGNAGE_NOTE`)했고, green은 C 표지판이 확인되기 전까지 pad memory를 신뢰하지 않는 보호장치를 둔다.

### 3) 목적지 패드 도착 → 360° 재스캔 + 시각 정밀 접근
기록된 pad 좌표로 대략 접근한 뒤, 패드가 카메라에 잡히지 않으면 제자리 360° 회전 재스캔(`_search_pad_xy_by_rotation`)으로 패드 blob의 bearing·거리를 다시 잡는다. 이후 scan + 전진을 반복하는 closed-loop visual creep(`_visual_creep_to_pad`)로 place 사정권 안까지 다가간다. 패드는 고정이고 로봇 자세만 임의이므로, 안 보이면 돌아서 스캔하는 것이 정책이다.

### 4) 배송 성공 pad pose 메모리 캐싱 → 같은 색 재배송 시 직행
place에 성공한 위치(`_PLACE_SUCCESS_POSES`)와 관측된 pad center(`_PAD_POSITIONS`)를 메모리에 저장한다. 같은 색 큐브를 다시 배송할 때는 이 캐시된 pose로 곧바로 접근 — 재탐색 비용을 줄여 처리량을 확보한다. 단, 시각 정밀 접근이 실패하면 해당 pose만 무효화하고 다시 센싱부터 시작해 잘못된 좌표가 누적되지 않도록 한다.

> 결정 루프: 매 cycle 텍스트 LLM이 고수준 행동(pick / deliver / recover / skip / stop)을 선택하면, 결정론적 실행 함수가 저수준 이동·pick·place를 수행한다. 고수준 추론은 LLM, 실시간 제어는 결정론 — 속도·안정성과 LLM 추론을 겸하는 표준 분업 구조. LLM 호출 장애 시에만 규칙 fallback이 안전망으로 동작한다.

In [ ]:
"""Level 1 본선 제출용 autonomous sorting agent — self-contained notebook cell.

이 셀 하나만 실행하면 에이전트가 동작합니다 (import-only 아님).
주석은 섹션을 명확히 분리해 알아보기 쉽게 작성했습니다.

아키텍처 한눈에 보기
====================
1. SUPPORT CODE — 과제 정의·LLM 결정 스키마·SDK 래퍼·데이터클래스.
   scene_state / 정답 entity ID / 정답 좌표 사용 금지.

2. SPEED CONTROL MODULE      (본선 허용 모듈 #2: 속도 제어 함수)
   - move_velocity / safe_turn_to_bearing / safe_lateral_offset
   - 속도·회전 파라미터는 이 모듈에 상수로 정의한다.

3. C-D LINE NAVIGATION MODULE (본선 허용 모듈 #1: C-D 라인 이동)
   - YELLOW_SHELF_NO_GO_RECTS (고정 장애물: 노란 선반) + _plan_safe_route (규칙 기반 후보 게이트 탐색).
   - B/C/D 패드 라인 이동 시 선반 관통 회피.

4. A/SOURCE 탐색 — 정답 좌표 없이 센싱만 사용.
   - _senses_conveyor_source: 인접 큐브가 합쳐진 큰 blob도 multi-color 군집으로 인식.
   - _ensure_at_conveyor_a: 제자리 360° 회전 센싱 스캔 + VLM A 표지판 1회 보조.

5. BLIND PICK — _close_in_and_blind_pick 이 수학 추정(bearing/distance)으로
   source에 close-in 후 pick_entity 로 가까운 큐브를 잡는다 (개별 큐브 분리 불필요).

6. DELIVERY — held_color → recorded pad memory 기준 검증 → C-D 경로 →
   visual creep → place → 성공 시 memory 기록 / 실패 시 invalidate.

실행 루프 (4단계 센싱 우선 — 위 전략 markdown과 대응)
====================================================
Step 1. 랜덤 시작 위치 → 제자리 360° 센싱 스캔으로 큐브/패드 표지/A 방향 동시 관측.
Step 2. 가장 가까운 큐브 pick 후, held 색 목적지 패드를 표지판 규칙 + VLM 프롬프트로 탐색.
Step 3. 목적지 패드에 도착하면 360° 회전 재스캔 + visual creep로 place 사정권까지 정밀 접근.
Step 4. place 성공 pose를 메모리에 캐싱 — 같은 색 재배송 시 캐시된 pose로 직행(재탐색 생략).

Level 1 규칙: scene_state·정확 entity ID·정답 좌표 사용 불가.
Coordinate go_to는 학생 시스템이 관찰로 추정/기록한 좌표에만 사용.
delivered_cube_ids·held_cube_info는 공식 지원 코드가 허용하는 채점/상태 추적용.

모델: LLM/VLM 모두 qwen/qwen3.6-35b-a3b (환경변수 MENLO_LLM_MODEL·MENLO_VLM_MODEL 로 override 가능).
"""

from __future__ import annotations


import asyncio
import base64
import json
import math
import os
import threading
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from menlo_runner.completion import (
    DEFAULT_MAX_DELIVERED_CUBES,
    CompletionConfig,
    CompletionTimeout,
    CompletionTracker,
    completion_config_for_round,
)
from menlo_runner.llm import ask_vlm
from menlo_runner.perception import detect_color_blobs, jpeg_dimensions, resize_jpeg
from menlo_runner.scene import delivered_cube_ids, held_cube_info


# ---------------------------------------------------------------------------
# 지원 코드: 공통 과제 정의와 필수 LLM 결정 형식
# ---------------------------------------------------------------------------
# 과제 문장은 고정합니다. 목표는 cube 색상 순서와 시작 위치가 달라져도
# 소스 코드 변경 없이 처리하는 하나의 agent를 만드는 것입니다.
TASK = "Find and sort cubes from the source area into their matching destination pads."

# Notebook/Python starter에서 사용할 LLM 모델 선택입니다.
# 이 값을 바꾸거나 실행 전에 환경 변수/.env의 MENLO_LLM_MODEL을 설정하세요.
APPROVED_LLM_MODELS = (
    "minimaxai/minimax-m3",
    "qwen/qwen3.6-35b-a3b",
)
LLM_MODEL = os.environ.setdefault("MENLO_LLM_MODEL", "qwen/qwen3.6-35b-a3b")
VLM_MODEL = os.environ.setdefault("MENLO_VLM_MODEL", "qwen/qwen3.6-35b-a3b")

# 고정 표지판 정보는 사용할 수 있습니다. 단, 이를 정확한 coordinate나 entity ID로
# 바꾸지 말고 관찰을 해석하는 데만 사용하세요.
DESTINATION_SIGN_RULES = {
    "red": "B",
    "green": "C",
    "blue": "D",
    "yellow": "E",
}
# Level 1 최종평가는 4색 전체 배송이 핵심입니다.
# B/C는 장애물과 A 표지판 혼동 위험이 있어 더 조심하되, 더 이상 버리지 않습니다.
TARGET_COLORS = frozenset(DESTINATION_SIGN_RULES.keys())
SKIP_COLORS = frozenset()
SIGNAGE_NOTE = (
    "A는 conveyor/cube source area이며 destination이 아닙니다. "
    "A 주변의 초록 표지는 destination C가 아니므로 무시합니다. "
    "Destination sign은 B red, C green, D blue, E yellow입니다."
)

DETECTION_MAX_SIDE = 416
VLM_MAX_SIDE = 720
VLM_JPEG_QUALITY = 82
SOURCE_RING_RADIUS_M = 1.2
REFLECTION_FRONT_OFFSET_M = 0.75
REFLECTION_LEFT_OFFSET_M = 0.35
REFLECTION_RETRY_FRONT_OFFSET_M = 0.65
REFLECTION_RETRY_RIGHT_OFFSET_M = 0.25
CUBE_PICK_AREA_RATIO = 0.010
PAD_PLACE_AREA_RATIO = 0.040
PAD_CREEP_SCAN_YAWS = (-0.25, 0.0, 0.25)
SCAN_HEAD_YAWS = (-1.0, -0.5, 0.0, 0.5, 1.0)
BODY_TURN_BACK_WZ = 0.5
BODY_TURN_BACK_DURATION_S = 3.1
GREEN_PAD_REQUIRES_SIGN_CONFIRMATION = True
MIN_PAD_DISTANCE_FROM_A = 2.5
# pad memory는 VLM/vision으로 찾거나 place 성공 후 기록한 pose만 사용한다.
# 정답 pad 좌표 상수는 두지 않는다 — 오배송 검증은 recorded pad memory 기준.
DELIVER_ANCHOR_MATCH_MARGIN_M = 0.20
DELIVER_ANCHOR_MAX_RANGE_M = 1.85
CUBE_LIKE_MAX_BBOX_RATIO = 0.20
CUBE_NAV_MAX_RAW_DISTANCE_M = 2.35
CUBE_NAV_MAX_STEP_M = 1.05
CUBE_NAV_MAX_BEARING_DEG = 65.0
CUBE_LIKE_MIN_ASPECT = 0.55
CUBE_LIKE_MAX_ASPECT = 1.85
PICK_VISIBLE_MAX_BEARING_DEG = 45.0
PICK_VISIBLE_MIN_AREA = 2500
PICK_VISIBLE_MIN_AREA_RATIO = 0.006
VLM_GUIDE_MAX_TURN_DEG = 45.0
SAFE_WORLD_BOUNDS = (-4.25, 3.45, -6.75, 6.65)
# A/source 탐색: 제자리 360° 회전 센싱 스캔 설정 (VLM A-sign 보조 1회).
SENSING_A_SCAN_STEPS = 6
SENSING_A_SCAN_WZ = 0.55
SENSING_A_SCAN_STEP_DURATION_S = 1.4


@dataclass(frozen=True)
class ShelfNoGoRect:
    """Level 1에서 허용된 고정 장애물 상수: 노란 선반 진입 금지 구역."""

    name: str
    min_x: float
    max_x: float
    min_y: float
    max_y: float
    clearance_m: float = 0.30


# ===========================================================================
# C-D LINE NAVIGATION MODULE  (허용 모듈 #1: 고정 장애물 no-go zone)
#   - 노란 선반은 simulator go_to obstacle map에서 누락되는 경우가 있어
#     B/C/D 패드 라인 이동 중 관통을 직접 막는다.
#   - 본선 규칙: C-D 라인 이동 모듈 안에서만 고정 장애물 좌표를 둔다.
# ===========================================================================
YELLOW_SHELF_NO_GO_RECTS: tuple[ShelfNoGoRect, ...] = (
    ShelfNoGoRect("shelf_1", -1.70, -1.18, -0.05, 0.65, 0.08),
    ShelfNoGoRect("shelf_2", -0.72, -0.22, -0.05, 0.65, 0.08),
    ShelfNoGoRect("shelf_3", -0.22, 0.28, -0.05, 0.65, 0.08),
    ShelfNoGoRect("shelf_4", 0.28, 0.72, -0.05, 0.65, 0.08),
)
SHELF_ROUTE_GATE_MARGIN_M = 0.75

# LLM은 아래 set에서 상위 단계 행동을 선택해야 합니다. 원시 속도 명령을
# 직접 출력하지 말고, 결정적 코드가 결정을 robot 행동으로 변환해야 합니다.
ALLOWED_NEXT_ACTIONS = {
    "search_cube",
    "navigate_to_cube",
    "pick_cube",
    "search_pad",
    "navigate_to_pad",
    "place_cube",
    "recover",
    "skip_target",
    "stop",
}


def _fresh_color_memory() -> dict[str, dict[str, Any]]:
    return {
        color: {
            "destination_sign": sign,
            "pad_center": None,
            "pad_approach_pose": None,
            "place_success_pose": None,
            "last_sign_bearing": None,
            "pick_failures": 0,
            "place_failures": 0,
            "success_count": 0,
        }
        for color, sign in DESTINATION_SIGN_RULES.items()
    }


@dataclass
class AgentDecision:
    """LLM이 반환하고 코드가 검증한 상위 단계 결정입니다."""

    next_action: str
    target_color: str | None = None
    reason: str = ""
    recovery_strategy: str | None = None


@dataclass
class AgentMemory:
    """observe-decide-act cycle 사이에 agent가 유지하는 상태입니다."""

    delivered_count: int = 0
    held_color: str | None = None
    active_color: str | None = None
    stage: str = "need_cube"
    search_turns: int = 0
    failed_attempts: dict[str, int] = field(default_factory=dict)
    pad_positions: dict[str, tuple[float, float]] = field(default_factory=dict)
    pad_approach_poses: dict[str, tuple[float, float]] = field(default_factory=dict)
    place_success_poses: dict[str, tuple[float, float]] = field(default_factory=dict)
    last_sign_bearing: dict[str, float] = field(default_factory=dict)
    recent_a_position: tuple[float, float] | None = None
    decision_source: str = "deterministic"
    last_llm_decision: dict[str, Any] | None = None
    color_memory: dict[str, dict[str, Any]] = field(default_factory=_fresh_color_memory)
    completed_colors: list[str] = field(default_factory=list)
    skipped_colors: list[str] = field(default_factory=list)
    last_pad_target: str | None = None
    logs: list[dict[str, Any]] = field(default_factory=list)


@dataclass
class Observation:
    """LLM과 실행 코드에 전달할 간결한 관찰입니다."""

    robot_status: Any
    detections: list[Any]
    note: str = ""
    vlm_summary: str = ""


@dataclass(frozen=True)
class ScannedDetection:
    """해당 camera frame을 얻을 때 사용한 head pose가 함께 기록된 color detection입니다."""

    color: str
    angle_deg: float
    blob_area: int
    centroid: tuple[int, int]
    bbox: tuple[int, int, int, int]
    head_yaw: float
    head_pitch: float
    image_size: tuple[int, int] | None = None
    area_ratio: float | None = None

    @property
    def full_bearing_deg(self) -> float:
        """대략적인 body-relative bearing입니다. Image angle에 head yaw를 더합니다."""
        return self.angle_deg + math.degrees(self.head_yaw)


def parse_agent_decision(text: str) -> AgentDecision | None:
    """필수 structured LLM JSON output을 parse하고 validate합니다."""
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = stripped.strip("`")
        if stripped.lower().startswith("json"):
            stripped = stripped[4:].strip()
    if not stripped.startswith("{"):
        start = stripped.find("{")
        end = stripped.rfind("}")
        if start >= 0 and end > start:
            stripped = stripped[start : end + 1]
    try:
        data = json.loads(stripped)
    except json.JSONDecodeError:
        return None

    next_action = data.get("next_action")
    if next_action not in ALLOWED_NEXT_ACTIONS:
        return None

    target_color = data.get("target_color")
    if target_color is not None and not isinstance(target_color, str):
        return None

    return AgentDecision(
        next_action=next_action,
        target_color=target_color,
        reason=str(data.get("reason", "")),
        recovery_strategy=data.get("recovery_strategy"),
    )


def build_decision_context(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Robot state를 LLM에 전달하기 좋은 간결한 text context로 변환합니다."""
    visible = [
        {
            "color": detection.color,
            "angle_deg": detection.angle_deg,
            "full_bearing_deg": round(getattr(detection, "full_bearing_deg", detection.angle_deg), 1),
            "blob_area": detection.blob_area,
            "area_ratio": round(float(getattr(detection, "area_ratio", 0.0) or 0.0), 6),
            "bbox": detection.bbox,
        }
        for detection in observation.detections
    ]
    return {
        "task": task,
        "visible_targets": visible,
        "held_color": memory.held_color,
        "active_color": memory.active_color,
        "stage": memory.stage,
        "delivered_count": memory.delivered_count,
        "completed_colors": memory.completed_colors,
        "skipped_colors": memory.skipped_colors,
        "failed_attempts": memory.failed_attempts,
        "pad_positions": memory.pad_positions,
        "pad_approach_poses": memory.pad_approach_poses,
        "place_success_poses": memory.place_success_poses,
        "last_sign_bearing": memory.last_sign_bearing,
        "color_memory": memory.color_memory,
        "recent_a_position": memory.recent_a_position,
        "decision_source": memory.decision_source,
        "last_llm_decision": memory.last_llm_decision,
        "last_result": last_result,
        "note": observation.note,
        "signage_note": SIGNAGE_NOTE,
        "vlm_summary": observation.vlm_summary,
    }


# ---------------------------------------------------------------------------
# 지원 코드: project 규칙에 맞는 SDK wrapper
# ---------------------------------------------------------------------------
# 이 래퍼들은 프로젝트 규칙에 맞는 input을 노출합니다. 아래 progress helper는
# completion과 robot이 cube를 들고 있는지 추적할 수 있도록 허용됩니다.
# Ground-truth coordinate, 정확한 target ID, global asset map은 추가하지 마세요.

async def get_robot_status(ctx: Any) -> Any:
    """Robot pose, motion status, neck state를 읽습니다.
    ctx.state()는 SDK 자체에 timeout이 없어 sim 연결이 끊기면 영원히 멈출 수 있으므로
    30s 타임아웃을 걸어 TimeoutError로 변환한다 (호출부의 try/except가 복구 가능하게)."""
    t0 = time.perf_counter()
    try:
        result = await asyncio.wait_for(ctx.state("robot_status"), timeout=30.0)
    except Exception as exc:
        elapsed = time.perf_counter() - t0
        _log_jsonl({
            "event": "robot_status_error",
            "error_type": type(exc).__name__,
            "error": str(exc),
            "seconds": round(elapsed, 4),
        })
        raise
    elapsed = time.perf_counter() - t0
    _log_jsonl({"event": "skill_latency", "skill": "robot_status", "seconds": round(elapsed, 4)})
    _log_robot_speed(result)
    _log_robot_joints(result)
    return result


async def get_camera_frame(ctx: Any) -> bytes:
    """POV camera frame을 가져옵니다. 동일한 이유로 30s 타임아웃을 건다."""
    t0 = time.perf_counter()
    result = await asyncio.wait_for(ctx.get_vision("pov"), timeout=30.0)
    elapsed = time.perf_counter() - t0
    _log_jsonl({"event": "skill_latency", "skill": "camera", "seconds": round(elapsed, 4)})
    return result


async def get_delivered_count(ctx: Any) -> int:
    """공통 workshop progress helper로 delivered cube 수를 셉니다."""
    ids = await asyncio.wait_for(delivered_cube_ids(ctx), timeout=20.0)
    return len(ids)


async def get_held_cube_info(ctx: Any) -> dict[str, str] | None:
    """Robot이 cube를 들고 있으면 현재 held cube id/color를 반환합니다."""
    held = await asyncio.wait_for(held_cube_info(ctx), timeout=20.0)
    return {"entity_id": held[0], "color": held[1]} if held else None


def build_signage_vlm_prompt(held_color: str | None = None) -> str:
    """고정 warehouse signage를 읽기 위한 strategy-neutral prompt를 만듭니다."""
    target = ""
    if held_color in DESTINATION_SIGN_RULES:
        target = f" Robot이 {held_color} cube를 들고 있으므로 target destination sign은 {DESTINATION_SIGN_RULES[held_color]}입니다."
    return (
        "이 robot camera frame에 보이는 warehouse sign을 읽으세요. "
        f"{SIGNAGE_NOTE} "
        "보이는 sign letter, color, 대략적인 left/center/right 위치, confidence를 JSON으로 반환하세요."
        + target
    )


def build_conveyor_a_vlm_prompt() -> str:
    """A 컨베이어(큐브 공급) 표지판 방향을 찾기 위한 prompt. A는 destination이 아님을 명시."""
    return (
        "이 robot camera frame에서 'A' 글자 표지판(conveyor/cube source)을 찾으세요. "
        f"{SIGNAGE_NOTE} "
        "A 표지판이 보이면 대략적인 left/center/right 위치와 sign이 frame의 좌·우 어디에 있는지 JSON으로 반환하세요. "
        '예: {"a_visible": true, "a_position": "left|center|right", "confidence": 0.0-1.0}'
    )


def _parse_a_bearing_from_text(text: str) -> float | None:
    """VLM 텍스트에서 A 표지판의 대략적 bearing(deg) 추정. left→-60, center→0, right→+60."""
    if not text:
        return None
    low = text.lower()
    if '"a_visible"' in low and 'false' in low.split('"a_visible"', 1)[1][:30]:
        return None
    if "left" in low:
        return -60.0
    if "right" in low:
        return 60.0
    if "center" in low or "centre" in low:
        return 0.0
    return None


def _parse_destination_sign_visible(text: str, expected_letter: str) -> bool:
    """VLM 응답에 목적지 표지판 글자(B/C/D/E)가 보이는지 확인."""
    if not text or not expected_letter:
        return False
    letter = expected_letter.upper()
    if letter == "A":
        return False
    low = text.lower()
    # 목적지 글자가 명시적으로 보이면 OK
    markers = (
        f'"{letter.lower()}"',
        f"'{letter.lower()}'",
        f"sign {letter.lower()}",
        f"letter {letter.lower()}",
        f"destination {letter.lower()}",
        f"pad {letter.lower()}",
        f" {letter.lower()} sign",
        f"target destination sign is {letter.lower()}",
    )
    return any(m in low for m in markers)


def _dist_xy(a: tuple[float, float], b: tuple[float, float]) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])


def _rect_bounds(rect: ShelfNoGoRect) -> tuple[float, float, float, float]:
    return (
        rect.min_x - rect.clearance_m,
        rect.max_x + rect.clearance_m,
        rect.min_y - rect.clearance_m,
        rect.max_y + rect.clearance_m,
    )


def _point_in_rect(x: float, y: float, rect: ShelfNoGoRect) -> bool:
    min_x, max_x, min_y, max_y = _rect_bounds(rect)
    return min_x <= x <= max_x and min_y <= y <= max_y


def _point_in_no_go(x: float, y: float) -> bool:
    """노란 선반 금지 구역 내부인지 확인한다."""
    return any(_point_in_rect(float(x), float(y), rect) for rect in YELLOW_SHELF_NO_GO_RECTS)


def _point_in_safe_world(x: float, y: float) -> bool:
    min_x, max_x, min_y, max_y = SAFE_WORLD_BOUNDS
    return min_x <= float(x) <= max_x and min_y <= float(y) <= max_y


def _segment_intersects_rect(
    a: tuple[float, float],
    b: tuple[float, float],
    rect: ShelfNoGoRect,
) -> bool:
    """Liang-Barsky clipping으로 선분-사각형 교차를 판정한다."""
    min_x, max_x, min_y, max_y = _rect_bounds(rect)
    x0, y0 = float(a[0]), float(a[1])
    x1, y1 = float(b[0]), float(b[1])
    if min_x <= x0 <= max_x and min_y <= y0 <= max_y:
        return True
    if min_x <= x1 <= max_x and min_y <= y1 <= max_y:
        return True

    dx = x1 - x0
    dy = y1 - y0
    u1, u2 = 0.0, 1.0
    for p, q in (
        (-dx, x0 - min_x),
        (dx, max_x - x0),
        (-dy, y0 - min_y),
        (dy, max_y - y0),
    ):
        if abs(p) < 1e-9:
            if q < 0:
                return False
            continue
        r = q / p
        if p < 0:
            if r > u2:
                return False
            if r > u1:
                u1 = r
        else:
            if r < u1:
                return False
            if r < u2:
                u2 = r
    return u1 <= u2


def _first_blocked_rect(points: list[tuple[float, float]]) -> ShelfNoGoRect | None:
    for point in points:
        for rect in YELLOW_SHELF_NO_GO_RECTS:
            if _point_in_rect(point[0], point[1], rect):
                return rect
    for start, end in zip(points, points[1:]):
        for rect in YELLOW_SHELF_NO_GO_RECTS:
            if _segment_intersects_rect(start, end, rect):
                return rect
    return None


def _path_is_clear(points: list[tuple[float, float]]) -> bool:
    return _first_blocked_rect(points) is None


def _nearest_clear_point(x: float, y: float) -> tuple[float, float]:
    """금지 구역 내부 target을 가장 가까운 바깥 점으로 밀어낸다."""
    px, py = float(x), float(y)
    step_out = 0.05
    for _ in range(len(YELLOW_SHELF_NO_GO_RECTS) + 2):
        moved = False
        for rect in YELLOW_SHELF_NO_GO_RECTS:
            if not _point_in_rect(px, py, rect):
                continue
            min_x, max_x, min_y, max_y = _rect_bounds(rect)
            options = (
                (abs(px - min_x), (min_x - step_out, py)),
                (abs(px - max_x), (max_x + step_out, py)),
                (abs(py - min_y), (px, min_y - step_out)),
                (abs(py - max_y), (px, max_y + step_out)),
            )
            _, (px, py) = min(options, key=lambda item: item[0])
            moved = True
        if not moved or not _point_in_no_go(px, py):
            break
    return (float(px), float(py))


def _shelf_route_gate_candidates() -> list[tuple[str, tuple[float, float]]]:
    """선반 union box 주위의 우회 gate 후보를 만든다."""
    if not YELLOW_SHELF_NO_GO_RECTS:
        return []
    bounds = [_rect_bounds(rect) for rect in YELLOW_SHELF_NO_GO_RECTS]
    min_x = min(b[0] for b in bounds)
    max_x = max(b[1] for b in bounds)
    min_y = min(b[2] for b in bounds)
    max_y = max(b[3] for b in bounds)
    margin = SHELF_ROUTE_GATE_MARGIN_M
    mid_x = (min_x + max_x) / 2.0
    mid_y = (min_y + max_y) / 2.0
    raw = [
        ("source_side_low", (min_x - margin, min_y - margin)),
        ("source_side_high", (min_x - margin, max_y + margin)),
        ("far_side_low", (max_x + margin, min_y - margin)),
        ("far_side_high", (max_x + margin, max_y + margin)),
        ("d_side", (min_x - margin, mid_y)),
        ("c_side", (max_x + margin, mid_y)),
        ("front_mid", (mid_x, min_y - margin)),
        ("back_mid", (mid_x, max_y + margin)),
    ]
    seen: set[tuple[int, int]] = set()
    out: list[tuple[str, tuple[float, float]]] = []
    for name, xy in raw:
        key = (round(xy[0] * 100), round(xy[1] * 100))
        if key in seen or _point_in_no_go(xy[0], xy[1]):
            continue
        seen.add(key)
        out.append((name, (float(xy[0]), float(xy[1]))))
    return out


def _route_distance(points: list[tuple[float, float]]) -> float:
    return sum(_dist_xy(a, b) for a, b in zip(points, points[1:]))


# --- C-D LINE NAVIGATION MODULE: no-go waypoint routing (규칙 기반 후보 경로 탐색) ---
def _plan_safe_route(
    start_xy: tuple[float, float],
    target_xy: tuple[float, float],
    *,
    purpose: str = "go_to",
    color: str | None = None,
) -> dict[str, Any]:
    """노란 선반 금지 구역을 피하는 waypoint 경로를 만든다."""
    start = (float(start_xy[0]), float(start_xy[1]))
    target = (float(target_xy[0]), float(target_xy[1]))
    original_target = target
    escape_prefix: list[tuple[float, float]] = []
    start_for_plan = start
    escape = False
    if _point_in_no_go(start[0], start[1]):
        start_for_plan = _nearest_clear_point(start[0], start[1])
        escape_prefix.append(start_for_plan)
        escape = True

    target_adjusted = False
    if _point_in_no_go(target[0], target[1]):
        target = _nearest_clear_point(target[0], target[1])
        target_adjusted = True

    direct_after = [target]
    if _path_is_clear([start_for_plan, *direct_after]):
        waypoints = [*escape_prefix, *direct_after]
        return {
            "waypoints": waypoints,
            "route_source": "escape_no_go" if escape else ("target_adjusted" if target_adjusted else "direct"),
            "shelf_avoidance_applied": escape or target_adjusted,
            "blocked_rect": None,
            "original_target": original_target,
            "target_adjusted": target_adjusted,
            "purpose": purpose,
            "color": color,
            "route_distance_m": _route_distance([start, *waypoints]),
        }

    blocked = _first_blocked_rect([start_for_plan, target])
    gates = _shelf_route_gate_candidates()
    candidates: list[tuple[float, float, str, list[tuple[float, float]]]] = []
    for name, gate in gates:
        after = [gate, target]
        if _path_is_clear([start_for_plan, *after]):
            waypoints = [*escape_prefix, *after]
            distance = _route_distance([start, *waypoints])
            candidates.append((distance, distance, name, waypoints))
    for name1, gate1 in gates:
        for name2, gate2 in gates:
            if name1 == name2:
                continue
            after = [gate1, gate2, target]
            if _path_is_clear([start_for_plan, *after]):
                waypoints = [*escape_prefix, *after]
                source_bias = 0.0 if name1.startswith("source_side") else 0.01
                distance = _route_distance([start, *waypoints])
                candidates.append((distance + source_bias, distance, f"{name1}->{name2}", waypoints))

    if candidates:
        _, distance, route_source, waypoints = min(candidates, key=lambda item: item[0])
        return {
            "waypoints": waypoints,
            "route_source": route_source,
            "shelf_avoidance_applied": True,
            "blocked_rect": blocked.name if blocked else None,
            "original_target": original_target,
            "target_adjusted": target_adjusted,
            "purpose": purpose,
            "color": color,
            "route_distance_m": round(float(distance), 3),
        }

    return {
        "waypoints": [],
        "route_source": "no_safe_route",
        "shelf_avoidance_applied": True,
        "blocked_rect": blocked.name if blocked else None,
        "original_target": original_target,
        "target_adjusted": target_adjusted,
        "purpose": purpose,
        "color": color,
        "route_distance_m": None,
    }


def _plan_safe_waypoints(
    start_xy: tuple[float, float],
    target_xy: tuple[float, float],
    *,
    purpose: str = "go_to",
    color: str | None = None,
) -> list[tuple[float, float]]:
    return list(_plan_safe_route(start_xy, target_xy, purpose=purpose, color=color)["waypoints"])


def _safe_pad_approach_pose(
    robot_xy: tuple[float, float],
    pad_center: tuple[float, float],
    *,
    retry: bool = False,
) -> tuple[float, float]:
    """반사각 접근 pose가 선반 안쪽이면 반대 offset 또는 가까운 clear point로 보정한다."""
    approach = _reflection_approach_pose(robot_xy, pad_center, retry=retry)
    if not _point_in_no_go(approach[0], approach[1]):
        return approach
    alt = _reflection_approach_pose(robot_xy, pad_center, retry=not retry)
    if not _point_in_no_go(alt[0], alt[1]):
        return alt
    return _nearest_clear_point(approach[0], approach[1])


def _area_ratio(det: Any) -> float:
    """blob_area/image_area. 압축 해상도가 바뀌어도 후보 크기를 비교하기 위한 값."""
    explicit = getattr(det, "area_ratio", None)
    if explicit is not None:
        try:
            return float(explicit)
        except Exception:
            pass
    image_size = getattr(det, "image_size", None)
    try:
        width, height = image_size
        return float(getattr(det, "blob_area", 0) or 0) / float(max(1, int(width) * int(height)))
    except Exception:
        return 0.0


def _world_xy_from_detection(robot_status: Any, det: Any, distance_m: float) -> tuple[float, float]:
    """robot pose + body-relative bearing + distance → world xy."""
    rx, ry = _robot_xy(robot_status)
    yaw = _robot_yaw_rad(robot_status)
    bearing_body = math.radians(getattr(det, "full_bearing_deg", getattr(det, "angle_deg", 0.0)))
    world_bearing = yaw + bearing_body
    return (
        float(rx + distance_m * math.cos(world_bearing)),
        float(ry + distance_m * math.sin(world_bearing)),
    )


def _reflection_approach_pose(
    robot_xy: tuple[float, float],
    pad_center: tuple[float, float],
    *,
    retry: bool = False,
) -> tuple[float, float]:
    """pad 중심을 바로 찍지 않고 비스듬한 접근 pose를 계산한다.

    1차: center - 0.75*u + 0.35*left
    실패 재시도: center - 0.65*u - 0.25*left
    여기서 u는 robot→pad 단위벡터, left는 u의 좌측 법선벡터다.
    """
    ux = pad_center[0] - robot_xy[0]
    uy = pad_center[1] - robot_xy[1]
    norm = math.hypot(ux, uy)
    if norm < 1e-6:
        yaw = 0.0
        ux, uy = math.cos(yaw), math.sin(yaw)
    else:
        ux, uy = ux / norm, uy / norm
    left_x, left_y = -uy, ux
    if retry:
        front = REFLECTION_RETRY_FRONT_OFFSET_M
        lateral = -REFLECTION_RETRY_RIGHT_OFFSET_M
    else:
        front = REFLECTION_FRONT_OFFSET_M
        lateral = REFLECTION_LEFT_OFFSET_M
    return (
        float(pad_center[0] - front * ux + lateral * left_x),
        float(pad_center[1] - front * uy + lateral * left_y),
    )


def _too_close_to_a(x: float, y: float, *, margin: float = 1.5) -> bool:
    """컨베이어 A 근처면 destination place 금지."""
    if _A_POSITION is None:
        return False
    return _dist_xy((x, y), _A_POSITION) < margin


def _pad_blob_near_a(robot_status: Any, det: Any, *, margin: float = 2.2) -> bool:
    """패드 blob이 가리키는 world xy가 A 근처면 True (A 초록 표지 등 destination 아님)."""
    if _A_POSITION is None:
        return False
    try:
        raw_distance = _distance_from_pad_blob_area(int(getattr(det, "blob_area", 0) or 0))
        distance = max(raw_distance - 0.4, 0.6)
        tx, ty = _world_xy_from_detection(robot_status, det, distance)
    except Exception:
        return False
    return _too_close_to_a(float(tx), float(ty), margin=margin)


def _filter_pads_safe(
    detections: list[Any],
    target_color: str | None,
    robot_status: Any | None = None,
) -> list[Any]:
    """패드 blob 필터 + green일 때 A 근처 blob 제거."""
    pads = _filter_pads(detections, target_color)
    if target_color == "green" and robot_status is not None:
        pads = [p for p in pads if not _pad_blob_near_a(robot_status, p)]
    return pads


def _log_jsonl(record: dict[str, Any]) -> None:
    """구조화 JSONL 로그: 파일 append + 선택적 Loki push."""
    payload = dict(record)
    payload["ts"] = time.time()
    line = json.dumps(payload, ensure_ascii=False, default=str)
    if _LOG_FILE is not None:
        with _LOG_LOCK:
            _LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
            with _LOG_FILE.open("a", encoding="utf-8") as fh:
                fh.write(line + "\n")
    if _LOKI_URL and _LOKI_USER and _LOKI_API_KEY:
        _push_loki_background(payload)
    try:
        from menlo_runner.metrics import observe_log
        observe_log(payload)
    except Exception:
        pass


def _push_loki_background(record: dict[str, Any]) -> None:
    """Non-blocking Loki HTTP push (별도 스레드)."""
    def _push() -> None:
        try:
            labels = {
                "job": "menlo-robot",
                "event": str(record.get("event", "unknown")),
                "held_color": str(record.get("held_color") or "?"),
            }
            body = json.dumps({
                "streams": [{
                    "stream": labels,
                    "values": [[str(int(record["ts"] * 1e9)), json.dumps(record, default=str)]],
                }],
            }).encode("utf-8")
            url = _LOKI_URL.rstrip("/") + "/loki/api/v1/push"
            req = urllib.request.Request(url, data=body, method="POST")
            req.add_header("Content-Type", "application/json")
            cred = f"{_LOKI_USER}:{_LOKI_API_KEY}".encode()
            req.add_header("Authorization", "Basic " + base64.b64encode(cred).decode())
            urllib.request.urlopen(req, timeout=3)
        except Exception:
            pass

    threading.Thread(target=_push, daemon=True).start()


def _is_pad_sign_confirmed(color: str) -> bool:
    """Green/C는 VLM으로 C sign을 본 뒤에만 recorded memory를 신뢰한다."""
    if color != "green" or not GREEN_PAD_REQUIRES_SIGN_CONFIRMATION:
        return True
    return color in _PAD_SIGN_CONFIRMED


def _invalidate_pad_memory(color: str, reason: str, *, clear_sign: bool = False) -> None:
    """오염된 pad memory를 지우고 다음 cycle에서 재탐색하게 한다."""
    removed = {
        "center": _PAD_POSITIONS.pop(color, None),
        "approach": _PAD_APPROACH_POSES.pop(color, None),
        "success": _PLACE_SUCCESS_POSES.pop(color, None),
        "bearing": _LAST_SIGN_BEARING.pop(color, None),
    }
    if clear_sign:
        _PAD_SIGN_CONFIRMED.discard(color)
    if any(value is not None for value in removed.values()):
        _log_jsonl({
            "event": "pad_memory_invalidated",
            "color": color,
            "reason": reason,
            "removed": removed,
        })


def _green_memory_can_store(color: str, *, sign_confirmed: bool = False) -> bool:
    """Green pad memory는 VLM으로 C sign 확인 후에만 저장한다."""
    if color != "green" or not GREEN_PAD_REQUIRES_SIGN_CONFIRMATION:
        return True
    if sign_confirmed or color in _PAD_SIGN_CONFIRMED:
        _PAD_SIGN_CONFIRMED.add(color)
        return True
    _invalidate_pad_memory(color, "green_memory_without_c_sign")
    _log_jsonl({"event": "pad_memory_rejected", "color": color, "reason": "green C sign not confirmed"})
    return False


def _nearest_pad_color(x: float, y: float) -> str:
    """기록된 pad memory 중 가장 가까운 색. 미기록 시 빈 문자열."""
    recorded = {c: p for c, p in _PAD_POSITIONS.items() if c in TARGET_COLORS}
    if not recorded:
        return ""
    return min(recorded, key=lambda color: _dist_xy((x, y), recorded[color]))


def _deliver_xy_matches_held_color(held_color: str, x: float, y: float) -> bool:
    """recorded pad memory 기준으로 held 색-패드 매칭 오류(E↔C 등)를 차단.

    정답 pad 좌표 상수는 사용하지 않는다. 같은 색의 기록된 pad가 없으면(아직 탐색 전)
    측정/creep이 신뢰 근거이므로 통과시키고, 기록된 다른 색 pad에 현저히 더 가까우면 거부한다.
    """
    target = (float(x), float(y))
    recorded = {c: p for c, p in _PAD_POSITIONS.items() if c in TARGET_COLORS}
    held_pos = recorded.get(held_color)
    if held_pos is None:
        for _oc, op in recorded.items():
            if _dist_xy(target, op) < 0.6:
                return False
        return True
    dist_held = _dist_xy(target, held_pos)
    for _oc, op in recorded.items():
        if _oc == held_color:
            continue
        if _dist_xy(target, op) + DELIVER_ANCHOR_MATCH_MARGIN_M < dist_held:
            return False
    return True


def _pad_center_plausible(x: float, y: float, *, color: str | None = None) -> bool:
    """패드 중심/접근 pose가 A source ring 안쪽이면 destination 오기록으로 본다."""
    if not _point_in_safe_world(float(x), float(y)):
        return False
    if _too_close_to_a(x, y, margin=MIN_PAD_DISTANCE_FROM_A):
        return False
    if color is not None and not _deliver_xy_matches_held_color(color, x, y):
        return False
    return True


def _has_proven_pad_memory(color: str) -> bool:
    """성공 place 후 기록된 pose만 검증된 pad memory로 취급한다."""
    success = _PLACE_SUCCESS_POSES.get(color)
    return success is not None and _pad_center_plausible(success[0], success[1], color=color)


def _resolve_deliver_approach(
    color: str,
    robot_xy: tuple[float, float],
) -> tuple[tuple[float, float] | None, str]:
    """배달 go_to 목표를 고른다. place 성공 전에는 coordinate go_to를 생략한다."""
    if not _has_proven_pad_memory(color):
        return None, "vision_only"

    success = _PLACE_SUCCESS_POSES.get(color)
    if success is not None and _pad_center_plausible(success[0], success[1], color=color):
        return (float(success[0]), float(success[1])), "success_pose"

    center = _valid_recorded_pad(color)
    if center is None:
        return None, "none"

    approach = _safe_pad_approach_pose(robot_xy, center)
    if not _pad_center_plausible(approach[0], approach[1], color=color):
        return None, "none"
    return approach, "recorded_center_reflection"


def _valid_recorded_pad(color: str) -> tuple[float, float] | None:
    """기록된 패드 좌표가 A 근처(오기록)면 무효화."""
    if color == "green" and not _is_pad_sign_confirmed(color):
        _invalidate_pad_memory(color, "unconfirmed_green_recorded_pad")
        return None
    pos = _PAD_POSITIONS.get(color)
    if pos is None:
        return None
    if not _pad_center_plausible(pos[0], pos[1], color=color):
        _invalidate_pad_memory(color, "recorded_pad_too_close_to_a" if _too_close_to_a(pos[0], pos[1], margin=MIN_PAD_DISTANCE_FROM_A) else "recorded_pad_wrong_anchor")
        return None
    return pos


def _valid_recorded_approach(color: str) -> tuple[float, float] | None:
    """기록된 pad 접근 pose가 A 근처(오기록)면 무효화."""
    if color == "green" and not _is_pad_sign_confirmed(color):
        _invalidate_pad_memory(color, "unconfirmed_green_recorded_approach")
        return None
    success = _PLACE_SUCCESS_POSES.get(color)
    if success is not None and _pad_center_plausible(success[0], success[1], color=color):
        return success
    # vision-only approach는 deliver에서 robot pose 기준 재계산하므로 여기서는 사용하지 않는다.
    return None


def _store_pad_memory(
    color: str,
    *,
    center_xy: tuple[float, float] | None = None,
    approach_xy: tuple[float, float] | None = None,
    success_xy: tuple[float, float] | None = None,
    sign_bearing: float | None = None,
    sign_confirmed: bool = False,
) -> None:
    """Level 1 허용 memory: 관찰로 추정하거나 성공 후 직접 기록한 pose만 저장."""
    if color not in TARGET_COLORS:
        return
    if not _green_memory_can_store(color, sign_confirmed=sign_confirmed):
        return
    if center_xy is not None and _pad_center_plausible(center_xy[0], center_xy[1], color=color):
        _PAD_POSITIONS[color] = (float(center_xy[0]), float(center_xy[1]))
    if approach_xy is not None and _pad_center_plausible(approach_xy[0], approach_xy[1], color=color):
        _PAD_APPROACH_POSES[color] = (float(approach_xy[0]), float(approach_xy[1]))
    if success_xy is not None and _pad_center_plausible(success_xy[0], success_xy[1], color=color):
        _PLACE_SUCCESS_POSES[color] = (float(success_xy[0]), float(success_xy[1]))
    if sign_bearing is not None:
        _LAST_SIGN_BEARING[color] = float(sign_bearing)


def _sync_memory_views(memory: AgentMemory) -> None:
    """전역 관찰/성공 pose cache를 AgentMemory의 색상별 view로 복사."""
    memory.pad_positions = dict(_PAD_POSITIONS)
    memory.pad_approach_poses = dict(_PAD_APPROACH_POSES)
    memory.place_success_poses = dict(_PLACE_SUCCESS_POSES)
    memory.last_sign_bearing = dict(_LAST_SIGN_BEARING)
    for color, sign in DESTINATION_SIGN_RULES.items():
        slot = memory.color_memory.setdefault(color, {"destination_sign": sign})
        slot["destination_sign"] = sign
        slot["pad_center"] = _PAD_POSITIONS.get(color)
        slot["pad_approach_pose"] = _PAD_APPROACH_POSES.get(color)
        slot["place_success_pose"] = _PLACE_SUCCESS_POSES.get(color)
        slot["last_sign_bearing"] = _LAST_SIGN_BEARING.get(color)
        slot["sign_confirmed"] = color in _PAD_SIGN_CONFIRMED
        slot["pick_failures"] = int(memory.failed_attempts.get(f"{color}:pick", 0))
        slot["place_failures"] = int(memory.failed_attempts.get(f"{color}:place", memory.failed_attempts.get(color, 0)))
        slot["success_count"] = sum(1 for completed in memory.completed_colors if completed == color)


async def _vlm_confirms_destination(ctx: Any, held_color: str) -> bool:
    """들고 있는 색에 맞는 destination sign(B/C/D/E)이 보이는지 VLM 확인. A만 보이면 False."""
    expected = DESTINATION_SIGN_RULES.get(held_color or "")
    if not expected:
        return False
    try:
        raw = await ask_vlm_about_frame(ctx, build_signage_vlm_prompt(held_color), api_key=_tokamak_api_key())
        confirmed = _parse_destination_sign_visible(raw, expected)
        if confirmed:
            _PAD_SIGN_CONFIRMED.add(held_color)
            _log_jsonl({
                "event": "pad_sign_confirmed",
                "held_color": held_color,
                "expected_sign": expected,
            })
        return confirmed
    except Exception:
        return False


async def _confirm_destination_sign_quick(
    ctx: Any,
    held_color: str,
    *,
    yaws: tuple[float, ...] = (0.0,),
) -> bool:
    """Green/C memory gate용 짧은 VLM 확인. 실패하면 오래 붙잡지 않는다."""
    if _is_pad_sign_confirmed(held_color):
        return True
    for yaw in yaws:
        try:
            await set_head(ctx, yaw=yaw, pitch=0.15)
            await asyncio.sleep(0.1)
            if await _vlm_confirms_destination(ctx, held_color):
                return True
        except Exception as exc:
            _log_jsonl({
                "event": "pad_sign_confirm_failed",
                "held_color": held_color,
                "yaw": yaw,
                "error": str(exc),
            })
    return False


async def ask_vlm_about_frame(ctx: Any, prompt: str, *, api_key: str) -> str:
    """Project에서 허용되는 VLM helper로 현재 POV frame에 대해 질문합니다."""
    jpeg = resize_jpeg(await get_camera_frame(ctx), max_side=VLM_MAX_SIDE, quality=VLM_JPEG_QUALITY)
    return await asyncio.to_thread(ask_vlm, jpeg, prompt, api_key=api_key)


async def _detect_color_blobs_from_camera(ctx: Any) -> tuple[list[Any], tuple[int, int]]:
    """압축된 POV frame에서 color blob을 찾고, ratio 계산용 image size를 같이 반환."""
    jpeg = resize_jpeg(await get_camera_frame(ctx), max_side=DETECTION_MAX_SIDE)
    image_size = jpeg_dimensions(jpeg)
    return detect_color_blobs(jpeg), image_size


async def perceive(ctx: Any) -> list[Any]:
    """현재 camera frame에서 Workshop 2 color-blob detector를 실행합니다."""
    detections, _ = await _detect_color_blobs_from_camera(ctx)
    return detections


async def set_head(ctx: Any, *, yaw: float | None = None, pitch: float | None = None) -> Any:
    """Walking direction을 바꾸지 않고 camera 방향을 조정합니다."""
    args: dict[str, float] = {}
    if yaw is not None:
        args["yaw"] = yaw
    if pitch is not None:
        args["pitch"] = pitch
    return await ctx.invoke("set_head", args, timeout_s=90)


# ===========================================================================
# SPEED CONTROL MODULE  (허용 모듈 #2: 속도 제어 함수)
#   - 본선 규칙: 속도/회전 파라미터를 별도 모듈로 분리해 상수로 정의하는 것을 허용한다.
#   - move_velocity / safe_turn_to_bearing / safe_lateral_offset 이 이 모듈에 해당한다.
# ===========================================================================
async def move_velocity(
    ctx: Any,
    *,
    vx: float = 0.0,
    vy: float = 0.0,
    wz: float = 0.0,
    duration_s: float = 1.0,
) -> Any:
    """짧은 body-frame velocity command를 보낸 뒤 멈춥니다."""
    t0 = time.perf_counter()
    timeout_s = max(15.0, float(duration_s) + 12.0)
    try:
        result = await ctx.invoke(
            "set_velocity",
            {"vx": vx, "vy": vy, "wz": wz, "duration_s": duration_s},
            timeout_s=timeout_s,
        )
    except Exception:
        try:
            await ctx.invoke("cancel", {}, timeout_s=5)
        except Exception:
            pass
        raise
    elapsed = time.perf_counter() - t0
    _log_jsonl({"event": "skill_latency", "skill": "move_velocity", "seconds": round(elapsed, 4)})
    return result


async def safe_turn_to_bearing(
    ctx: Any,
    bearing_deg: float,
    *,
    tolerance_deg: float = 8.0,
    max_turn_deg: float = VLM_GUIDE_MAX_TURN_DEG,
) -> bool:
    """target bearing이 중앙으로 오도록 짧게 회전."""
    clipped_bearing = max(-max_turn_deg, min(max_turn_deg, float(bearing_deg)))
    if abs(clipped_bearing) <= tolerance_deg:
        return False
    wz = -0.4 if clipped_bearing > 0 else 0.4
    duration = min(1.05, 0.25 + abs(clipped_bearing) / 60.0)
    _log_jsonl({
        "event": "safe_motion",
        "kind": "turn",
        "target_bearing": bearing_deg,
        "clipped_bearing": round(clipped_bearing, 2),
        "wz": wz,
        "duration_s": round(duration, 3),
    })
    await move_velocity(ctx, wz=wz, duration_s=duration)
    return True


async def safe_nudge_forward(ctx: Any, *, duration_s: float = 0.7, vx: float = 0.22) -> Any:
    """visual creep용 짧은 전진."""
    _log_jsonl({"event": "safe_motion", "kind": "forward", "vx": vx, "duration_s": duration_s})
    return await move_velocity(ctx, vx=vx, duration_s=duration_s)


async def safe_lateral_offset(ctx: Any, *, left: bool = True, duration_s: float = 0.45, vy: float = 0.16) -> Any:
    """place 직전 좌우 오프셋 보정."""
    signed_vy = vy if left else -vy
    _log_jsonl({"event": "safe_motion", "kind": "lateral", "vy": signed_vy, "duration_s": duration_s})
    return await move_velocity(ctx, vy=signed_vy, duration_s=duration_s)


async def cancel_action(ctx: Any) -> Any:
    """현재 실행 중인 runtime action을 취소합니다."""
    return await ctx.invoke("cancel", {}, timeout_s=30)


async def pick_nearest_cube(ctx: Any) -> Any:
    """Code가 robot을 시각적으로 충분히 위치시킨 뒤 nearest cube를 집습니다."""
    return await ctx.invoke(
        "pick_entity",
        {"target": {"kind": "entity", "entity_id": "cube"}},
        timeout_s=300,
    )


async def place_nearest_zone(ctx: Any) -> Any:
    """Matching pad에 도달한 뒤 nearest zone에 place합니다."""
    return await ctx.invoke("place_entity", {}, timeout_s=300)


def result_summary(result: Any) -> dict[str, Any]:
    """SDK result를 log하기 쉬운 작은 dictionary로 변환합니다."""
    error = getattr(result, "error", None)
    status = getattr(result, "status", None)
    inner = getattr(result, "result", None)
    out: dict[str, Any] = {
        "status": str(status) if status is not None else None,
        "error": getattr(error, "message", None) if error else None,
    }
    if isinstance(inner, dict):
        out["held"] = inner.get("held")
        out["held_count"] = inner.get("held_count")
    return out


async def scan_head(
    ctx: Any,
    *,
    yaws: tuple[float, ...] = (-0.8, 0.0, 0.8),
    pitch: float = 0.15,
) -> list[Any]:
    """간단한 scan helper입니다. 더 나은 search 전략으로 교체할 수 있습니다.
    스로틀링 시 set_head가 막히면 해당 yaw를 건너뛰어 폭주를 막는다."""
    all_detections: list[Any] = []
    for yaw in yaws:
        try:
            await set_head(ctx, yaw=yaw, pitch=pitch)
        except Exception:
            continue
        await asyncio.sleep(0.2)
        try:
            dets, image_size = await _detect_color_blobs_from_camera(ctx)
        except Exception:
            continue
        for detection in dets:
            image_area = max(1, image_size[0] * image_size[1])
            all_detections.append(
                ScannedDetection(
                    color=detection.color,
                    angle_deg=detection.angle_deg,
                    blob_area=detection.blob_area,
                    centroid=detection.centroid,
                    bbox=detection.bbox,
                    head_yaw=yaw,
                    head_pitch=pitch,
                    image_size=image_size,
                    area_ratio=float(detection.blob_area) / float(image_area),
                )
            )
    return all_detections


async def scan_for_pick_cubes(ctx: Any) -> tuple[list[Any], bool]:
    """전방 head scan만 수행한다. 큰 회전은 VLM/A 탐색 단계에서 제한적으로 처리한다."""
    dets = await scan_head(ctx, yaws=SCAN_HEAD_YAWS, pitch=0.15)
    if not _visible_pick_candidates(dets):
        _log_jsonl({"event": "cube_scan_front_only", "reason": "no_close_cube_in_front"})
    return dets, False


# ---------------------------------------------------------------------------
# 학생 구현: API key, 패드 좌표 기록, 유틸
# ---------------------------------------------------------------------------
# 패드(B/C/D/E)는 고정이므로 place 성공 시 로봇 자세를 기록해 재사용.
# 정답 좌표가 아닌 "직접 기록한 관측 pose"이므로 Level 1 규칙에 부합.
_PAD_POSITIONS: dict[str, tuple[float, float]] = {}
_PAD_APPROACH_POSES: dict[str, tuple[float, float]] = {}
_PLACE_SUCCESS_POSES: dict[str, tuple[float, float]] = {}
_LAST_SIGN_BEARING: dict[str, float] = {}
_PAD_SIGN_CONFIRMED: set[str] = set()
_A_POSITION: tuple[float, float] | None = None
_LOG_FILE: Path | None = None
_LOKI_URL: str | None = None
_LOKI_USER: str | None = None
_LOKI_API_KEY: str | None = None
_LOG_LOCK = threading.Lock()
_BASELINE_DELIVERED: int | None = None
_prev_pose: tuple[float, float] | None = None
_prev_pose_ts: float | None = None
_run_start_ts: float | None = None


def _run_delivered(total: int) -> int:
    """이번 scored run 기준 배달 수 (시작 baseline 제외)."""
    base = _BASELINE_DELIVERED if _BASELINE_DELIVERED is not None else 0
    return max(0, int(total) - base)


def _score(run_delivered: int) -> int:
    """최신 Level 1 점수: 첫 배송 60점, 이후 배송당 20점."""
    if run_delivered <= 0:
        return 0
    return 60 + (int(run_delivered) - 1) * 20


def _log_score(total_delivered: int) -> None:
    """run_delivered·score를 JSONL/Prometheus에 기록."""
    rd = _run_delivered(total_delivered)
    _log_jsonl({
        "event": "score",
        "run_delivered": rd,
        "score": _score(rd),
        "total_delivered": total_delivered,
    })


def _log_robot_speed(status: Any) -> None:
    """robot_status에서 속도(m/s)를 추정해 JSONL/Prometheus에 기록."""
    global _prev_pose, _prev_pose_ts
    try:
        rx, ry = _robot_xy(status)
        now = time.time()
        motion_speed: float | None = None
        pose_speed: float | None = None
        robot = getattr(status, "robot", None)
        motion = getattr(robot, "motion", None) if robot is not None else None
        if motion is not None:
            vx = float(getattr(motion, "vx", 0.0) or 0.0)
            vy = float(getattr(motion, "vy", 0.0) or 0.0)
            motion_speed = math.hypot(vx, vy)
        if _prev_pose is not None and _prev_pose_ts is not None:
            dt = now - _prev_pose_ts
            if dt > 0.05:
                dx = rx - _prev_pose[0]
                dy = ry - _prev_pose[1]
                pose_speed = math.hypot(dx, dy) / dt
        if motion_speed is not None and motion_speed > 1e-4:
            speed = motion_speed
            source = "motion"
        elif pose_speed is not None:
            speed = pose_speed
            source = "pose_delta"
        else:
            speed = motion_speed or 0.0
            source = "motion" if motion_speed is not None else "unknown"
        _prev_pose = (rx, ry)
        _prev_pose_ts = now
        _log_jsonl({
            "event": "robot_speed",
            "m_s": round(speed, 5),
            "source": source,
            "motion_m_s": round(motion_speed, 5) if motion_speed is not None else None,
            "pose_delta_m_s": round(pose_speed, 5) if pose_speed is not None else None,
        })
    except Exception:
        pass


def _log_throughput(total_delivered: int) -> None:
    """이번 run 기준 분당 배달 수(throughput)를 기록."""
    global _run_start_ts
    if _run_start_ts is None:
        _run_start_ts = time.time()
    elapsed = time.time() - _run_start_ts
    if elapsed < 1.0:
        return
    rd = _run_delivered(total_delivered)
    per_min = rd / (elapsed / 60.0)
    _log_jsonl({
        "event": "throughput",
        "per_min": round(per_min, 3),
        "run_delivered": rd,
        "score": _score(rd),
        "total_delivered": total_delivered,
    })


def _tokamak_api_key() -> str:
    import os

    key = os.environ.get("TOKAMAK_API_KEY")
    if key:
        return key
    try:
        from pathlib import Path as _Path
        from dotenv import load_dotenv

        for candidate in (_Path.cwd() / ".env", _Path.home() / ".env"):
            if candidate.exists():
                load_dotenv(candidate, override=False)
        key = os.environ.get("TOKAMAK_API_KEY")
        if key:
            return key
    except Exception:
        pass
    try:
        from menlo_runner.config import load_config

        return load_config(require_tokamak=True).tokamak_api_key
    except Exception as exc:
        raise RuntimeError("TOKAMAK_API_KEY를 찾을 수 없습니다.") from exc


def _robot_yaw_rad(robot_status: Any) -> float:
    """robot_status에서 yaw(라디안) 안전 추출."""
    try:
        pose = robot_status.robot.pose
        for attr in ("yaw", "yaw_rad", "heading"):
            val = getattr(pose, attr, None)
            if val is not None:
                return float(val)
        deg = getattr(pose, "yaw_deg", None)
        if deg is not None:
            return math.radians(float(deg))
    except Exception:
        pass
    return 0.0


def _robot_xy(robot_status: Any) -> tuple[float, float]:
    try:
        pos = robot_status.robot.pose.position
        return float(pos[0]), float(pos[1])
    except Exception:
        return 0.0, 0.0


def _robot_is_fallen(robot_status: Any) -> bool:
    """robot_status 에서 전도 여부 안전 탐지. 필드명 variation 대응."""
    try:
        for attr in ("fallen", "is_fallen", "tipped", "stability", "state"):
            val = getattr(robot_status, attr, None)
            if val is None:
                continue
            if isinstance(val, bool) and val:
                return True
            if isinstance(val, str) and val.lower() in {"fallen", "tipped", "down"}:
                return True
        inner = getattr(robot_status, "robot", None)
        for attr in ("fallen", "is_fallen", "tipped", "state"):
            val = getattr(inner, attr, None)
            if val is None:
                continue
            if isinstance(val, bool) and val:
                return True
            if isinstance(val, str) and val.lower() in {"fallen", "tipped", "down"}:
                return True
    except Exception:
        return False
    return False


def _extract_joint_values(robot_status: Any, *, limit: int = 32) -> dict[str, float]:
    """robot_status에 joint field가 있으면 숫자값만 작은 dict로 추출."""
    candidates: list[Any] = []
    try:
        robot = getattr(robot_status, "robot", None)
        candidates.extend([
            getattr(robot, "joints", None),
            getattr(robot, "joint_positions", None),
            getattr(robot, "joint_values", None),
            getattr(robot, "joint_state", None),
            getattr(robot_status, "joints", None),
            getattr(robot_status, "joint_positions", None),
        ])
    except Exception:
        pass

    def _coerce(value: Any) -> float | None:
        try:
            return float(value)
        except Exception:
            return None

    out: dict[str, float] = {}
    for item in candidates:
        if item is None:
            continue
        if isinstance(item, dict):
            iterable = item.items()
        elif isinstance(item, (list, tuple)):
            iterable = [(f"joint_{idx}", val) for idx, val in enumerate(item)]
        else:
            iterable = [
                (name, val)
                for name, val in vars(item).items()
                if not name.startswith("_")
            ] if hasattr(item, "__dict__") else []
        for name, val in iterable:
            numeric = _coerce(val)
            if numeric is None:
                numeric = _coerce(getattr(val, "position", None))
            if numeric is None:
                numeric = _coerce(getattr(val, "value", None))
            if numeric is None:
                continue
            out[str(name)] = round(numeric, 5)
            if len(out) >= limit:
                return out
        if out:
            return out
    return out


def _log_robot_joints(status: Any) -> None:
    joints = _extract_joint_values(status)
    if joints:
        _log_jsonl({"event": "robot_joints", "joints": joints})


def _distance_from_blob_area(blob_area: int) -> float:
    """blob_area → 거리(m) 추정. 큰 blob = 가까움. d = k/sqrt(area).
    보정: area≈8000px² → ~1m(집기 가능 거리)."""
    if blob_area <= 0:
        return 0.0
    k = math.sqrt(8000.0)  # ≈ 89.4
    return float(k) / math.sqrt(float(blob_area))


def _distance_from_pad_blob_area(blob_area: int) -> float:
    """패드/표지판 blob 전용 거리 추정. area≈50000px² → ~1.5m."""
    if blob_area <= 0:
        return 0.0
    k = math.sqrt(50000.0) * 1.5
    return float(k) / math.sqrt(float(blob_area))


def _bbox_wh(det: Any) -> tuple[int, int]:
    """detection bbox 에서 (width, height) 안전 추출. bbox=(x,y,w,h)."""
    try:
        bb = det.bbox
        if len(bb) >= 4:
            return int(bb[2]), int(bb[3])
    except Exception:
        pass
    return 0, 0


def _bbox_xy(det: Any) -> tuple[int, int]:
    try:
        bb = det.bbox
        if len(bb) >= 2:
            return int(bb[0]), int(bb[1])
    except Exception:
        pass
    return 0, 0


def _centroid_y_ratio(det: Any) -> float:
    image_size = getattr(det, "image_size", None)
    try:
        _, height = image_size
        if not height:
            return 0.5
        cy = int(getattr(det, "centroid")[1])
        return cy / float(height)
    except Exception:
        _, y = _bbox_xy(det)
        _, h = _bbox_wh(det)
        return 0.5 if h <= 0 else float(y + h / 2.0) / 416.0


def _is_cube_like(det: Any) -> bool:
    """큐브 판별: 작~중간 크기, 정사각형에 가까운 compact blob. 벽/표지판/패드 거대 blob 제외.
    L2 의 is_cube_like 와 동일 기준: area 300~14000, bbox w/h < 260, aspect 0.5~2.0."""
    try:
        area = int(det.blob_area)
    except Exception:
        return False
    if area < 180:
        return False
    w, h = _bbox_wh(det)
    if w <= 0 or h <= 0:
        return False
    aspect = w / h if h else 0
    if not (CUBE_LIKE_MIN_ASPECT <= aspect <= CUBE_LIKE_MAX_ASPECT):
        return False
    if _centroid_y_ratio(det) < 0.20:
        return False
    ratio = _area_ratio(det)
    if ratio > 0:
        image_size = getattr(det, "image_size", (0, 0))
        try:
            width, height = int(image_size[0]), int(image_size[1])
        except Exception:
            width, height = 0, 0
        max_bbox_ratio = 1.0
        if width and height:
            if w > width * 0.50:
                return False
            max_bbox_ratio = max(w / float(width), h / float(height))
        return 0.00025 <= ratio <= 0.038 and max_bbox_ratio <= CUBE_LIKE_MAX_BBOX_RATIO
    return 300 <= area <= 14000 and w <= 260 and h <= 260


def _filter_cubes(detections: list[Any], target_color: str | None = None) -> list[Any]:
    cands = [d for d in detections if _is_cube_like(d)]
    if target_color:
        cands = [d for d in cands if d.color == target_color]
    cands.sort(key=lambda d: d.blob_area, reverse=True)
    return cands


def _pick_preferred_cube(cubes: list[Any]) -> Any | None:
    """Pick 직전 후보 선택: 중앙에 가깝고 큰 큐브 우선."""
    if not cubes:
        return None

    def score(det: Any) -> float:
        bearing = abs(float(getattr(det, "full_bearing_deg", getattr(det, "angle_deg", 0.0)) or 0.0))
        area = max(1.0, float(getattr(det, "blob_area", 1.0) or 1.0))
        return bearing - min(area, 14000.0) / 14000.0 * 8.0

    return min(cubes, key=score)


def _filter_pads(detections: list[Any], target_color: str | None = None) -> list[Any]:
    """큰 바닥 패드 blob. 압축 frame에서는 area_ratio를 우선 사용한다."""
    def is_pad_like(det: Any) -> bool:
        try:
            area = int(getattr(det, "blob_area", 0) or 0)
        except Exception:
            return False
        w, h = _bbox_wh(det)
        if w <= 0 or h <= 0:
            return False
        aspect = w / float(h)
        if not 0.35 <= aspect <= 3.0:
            return False
        if _centroid_y_ratio(det) < 0.26:
            return False
        ratio = _area_ratio(det)
        if ratio > 0:
            image_size = getattr(det, "image_size", (0, 0))
            try:
                width, height = int(image_size[0]), int(image_size[1])
            except Exception:
                width, height = 0, 0
            if width and height and max(w / float(width), h / float(height)) > 0.55:
                return False
            return 0.025 <= ratio <= 0.13
        return area >= 15000

    cands = [d for d in detections if is_pad_like(d)]
    if target_color:
        cands = [d for d in cands if d.color == target_color]
    cands.sort(key=lambda d: (_area_ratio(d), getattr(d, "blob_area", 0)), reverse=True)
    return cands


# ---------------------------------------------------------------------------
# 학생 구현: LLM decision 함수
# ---------------------------------------------------------------------------
async def decide_next_action(
    task: str,
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """Text LLM을 사용해 다음 상위 단계 행동을 선택합니다.

    속도전(10분, 큐브당 20점)에서는 completion tracker가 시간 종료를 처리하므로
    여기서 인위적인 6개 stop을 걸지 않는다 — 계속 배송.
    """
    from menlo_runner.llm import call_llm

    decision_context = build_decision_context(task, observation, memory, last_result)

    system_prompt = (
        "Speed warehouse robot supervisor. 20pts/cube, maximize deliveries. "
        "red→B, green→C, blue→D, yellow→E. A=source only, never place at A. "
        "Actions: search_cube, navigate_to_cube, pick_cube, search_pad, navigate_to_pad, "
        "place_cube, recover, skip_target, stop. "
        "Holding→pad only. Not holding+cube visible→navigate_to_cube. Not holding+no cube→search_cube. "
        "Pick any color. Reply ONLY JSON fenced block:\n"
        '```json\n{"next_action":"...","target_color":"red|green|blue|yellow|null","reason":"..."}\n```'
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(decision_context, ensure_ascii=False, default=str)},
    ]

    try:
        reply = await asyncio.to_thread(call_llm, messages, api_key=_tokamak_api_key())
    except Exception as exc:
        return AgentDecision(next_action="recover", reason=f"LLM call failed: {exc}", recovery_strategy="llm_error")

    decision = parse_agent_decision(reply)
    if decision is None:
        return AgentDecision(next_action="recover", reason=f"unparseable: {reply[:160]}", recovery_strategy="bad_llm_output")

    # 결정적 보정: 들고 있는데 cube 계열이면 pad 계열로, 안 들고 있는데 pad 계열이면 cube 계열로
    holding = memory.held_color is not None
    cube_actions = {"search_cube", "navigate_to_cube", "pick_cube"}
    pad_actions = {"search_pad", "navigate_to_pad", "place_cube"}
    if holding and decision.next_action in cube_actions:
        return AgentDecision(next_action="search_pad", target_color=memory.held_color,
                             reason="holding cube → search matching pad", recovery_strategy=None)
    # 들고 있으면 target_color는 반드시 held_color (blue→D pad, red→B pad 등)
    if holding:
        expected_sign = DESTINATION_SIGN_RULES.get(memory.held_color or "", "?")
        recorded = _valid_recorded_approach(memory.held_color or "") or _valid_recorded_pad(memory.held_color or "")
        if recorded and decision.next_action in {"search_pad", "navigate_to_pad", "place_cube", "recover"}:
            return AgentDecision(next_action="navigate_to_pad", target_color=memory.held_color,
                                 reason=f"recorded approach for {memory.held_color}→{expected_sign}; direct navigate",
                                 recovery_strategy=None)
        if decision.next_action == "place_cube":
            return AgentDecision(next_action="search_pad", target_color=memory.held_color,
                                 reason=f"must reach {expected_sign} pad before place, not place at A",
                                 recovery_strategy=None)
        if decision.target_color and decision.target_color != memory.held_color:
            decision = AgentDecision(next_action=decision.next_action, target_color=memory.held_color,
                                     reason=f"force target={memory.held_color}→{expected_sign} pad",
                                     recovery_strategy=decision.recovery_strategy)
    if not holding and decision.next_action in pad_actions:
        return AgentDecision(next_action="search_cube", target_color=decision.target_color,
                             reason="not holding → search cube", recovery_strategy=None)
    # 속도전 강제: 큐브가 보이는데 search_cube 고르면 → 가장 큰 visible cube 로 navigate
    if not holding and decision.next_action == "search_cube" and observation.detections:
        cube_cands = [
            c for c in _filter_cubes(observation.detections)
            if c.color in TARGET_COLORS and 1200 <= c.blob_area <= 9000
        ]
        if cube_cands:
            best = _pick_preferred_cube(cube_cands) or cube_cands[0]
            return AgentDecision(next_action="navigate_to_cube", target_color=best.color,
                                 reason=f"target cube visible (area={best.blob_area}) → force navigate",
                                 recovery_strategy=None)
    return decision


def decide_next_action_rules(
    observation: Observation,
    memory: AgentMemory,
    last_result: dict[str, Any] | None = None,
) -> AgentDecision:
    """LLM 없이 즉시 다음 행동 결정. fallback/recovery 전용 (25–30s LLM 호출 생략)."""
    holding = memory.held_color is not None
    if holding and memory.held_color not in TARGET_COLORS:
        return AgentDecision(next_action="recover", reason="rules: unknown held color, recover")
    if holding:
        expected = DESTINATION_SIGN_RULES.get(memory.held_color or "", "?")
        recorded = _valid_recorded_approach(memory.held_color or "") or _valid_recorded_pad(memory.held_color or "")
        if recorded:
            return AgentDecision(
                next_action="navigate_to_pad",
                target_color=memory.held_color,
                reason=f"rules: recorded approach {memory.held_color}→{expected}",
            )
        pads = _filter_pads_safe(observation.detections, memory.held_color, observation.robot_status)
        if pads:
            obs = Observation(robot_status=observation.robot_status, detections=pads)
            xy = estimate_pad_xy_from_observation(obs, memory.held_color or "")
            if xy is not None:
                return AgentDecision(
                    next_action="navigate_to_pad",
                    target_color=memory.held_color,
                    reason=f"rules: visible {expected} pad, navigate",
                )
        if last_result and last_result.get("action") == "recover":
            return AgentDecision(
                next_action="search_pad",
                target_color=memory.held_color,
                reason=f"rules: keep searching {expected} pad after recovery",
            )
        return AgentDecision(
            next_action="search_pad",
            target_color=memory.held_color,
            reason=f"rules: rotate to find {expected} pad (ignore A green sign)",
        )

    cubes = [
        c for c in _filter_cubes(observation.detections)
        if c.color in TARGET_COLORS and 1200 <= c.blob_area <= 9000
    ]
    if cubes:
        best = _pick_preferred_cube(cubes) or cubes[0]
        return AgentDecision(
            next_action="navigate_to_cube",
            target_color=best.color,
            reason=f"rules: closest visible cube area={best.blob_area}",
        )
    if last_result and last_result.get("status") == "failed":
        return AgentDecision(next_action="recover", reason="rules: pick/search failed, recover")
    return AgentDecision(next_action="search_cube", reason="rules: no cube visible, search A")


# ---------------------------------------------------------------------------
# 학생 구현: observation, verification, memory
# ---------------------------------------------------------------------------
async def observe_world(ctx: Any, memory: AgentMemory, *, use_vlm: bool = False) -> Observation:
    """현재 observation 수집. 기본은 head scan만(VLM/LLM 경로 지연 최소화)."""
    robot_status = await get_robot_status(ctx)
    looking_for_pad = memory.held_color is not None

    if looking_for_pad:
        all_detections = await scan_head(ctx, yaws=(-1.0, -0.5, 0.0, 0.5, 1.0), pitch=0.15)
        pads = _filter_pads_safe(all_detections, memory.held_color, robot_status)
        cubes = _filter_cubes(all_detections)
        detections = pads + cubes
        note = f"searching pad for {memory.held_color}"
        vlm_summary = ""
        if use_vlm:
            try:
                vlm_summary = await ask_vlm_about_frame(
                    ctx, build_signage_vlm_prompt(memory.held_color), api_key=_tokamak_api_key(),
                )
            except Exception as exc:
                vlm_summary = f"VLM signage failed: {exc}"
        return Observation(robot_status=robot_status, detections=detections, note=note, vlm_summary=vlm_summary)

    all_detections = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
    detections = _filter_cubes(all_detections)
    return Observation(robot_status=robot_status, detections=detections, note="searching cube")


async def verify_outcome(ctx: Any, decision: AgentDecision, action_result: dict[str, Any]) -> dict[str, Any]:
    """마지막 action 성공 여부 확인. progress helper로 권위 있는 held/delivered 사용."""
    action = action_result.get("action")
    verified: dict[str, Any] = {"decision": decision.__dict__, "action_result": action_result}

    try:
        verified["delivered_count"] = await get_delivered_count(ctx)
    except Exception as exc:
        verified["delivered_count_error"] = str(exc)
    try:
        held = await get_held_cube_info(ctx)
        verified["held_cube"] = held
        verified["held_color"] = held["color"] if held else None
    except Exception as exc:
        verified["held_error"] = str(exc)

    if action == "pick_cube":
        res = action_result.get("result", {})
        # pick 성공 = status done AND 이제 cube를 들고 있음
        verified["picked_ok"] = res.get("status") == "done" and verified.get("held_color") is not None

    elif action == "place_cube":
        res = action_result.get("result", {})
        # place 성공 = status done AND 더 이상 들고 있지 않음
        verified["placed_ok"] = res.get("status") == "done" and verified.get("held_color") is None

    elif action in ("navigate_to_cube", "navigate_to_pad"):
        verified["reached"] = action_result.get("result", {}).get("status") == "done"
        # auto-pick / auto-place 통합: navigate 한 cycle 안에서 pick/place 수행 시 권위 metric 으로 성공 판정
        if action == "navigate_to_cube" and action_result.get("auto_pick"):
            verified["picked_ok"] = verified.get("held_color") is not None
        if action == "navigate_to_pad" and action_result.get("auto_place"):
            verified["placed_ok"] = verified.get("held_color") is None

    return verified


def update_memory(
    memory: AgentMemory,
    observation: Observation,
    decision: AgentDecision,
    verified: dict[str, Any],
) -> None:
    """각 cycle 뒤 지속 상태를 update합니다. 패드 위치 기록 포함."""
    global _A_POSITION
    action = verified.get("action_result", {}).get("action") or decision.next_action
    color = decision.target_color or memory.active_color

    # 권위 있는 delivered/held 우선
    if "delivered_count" in verified:
        try:
            memory.delivered_count = int(verified["delivered_count"])
        except Exception:
            pass
    if verified.get("held_color") is not None:
        memory.held_color = verified["held_color"]

    if action == "pick_cube" and verified.get("picked_ok"):
        memory.active_color = memory.held_color
        memory.stage = "need_pad"
        memory.failed_attempts.pop(memory.held_color, None)
        try:
            rx, ry = _robot_xy(observation.robot_status)
            _A_POSITION = (float(rx), float(ry))
        except Exception:
            pass

    elif action == "place_cube" and verified.get("placed_ok"):
        if memory.held_color and memory.held_color not in memory.completed_colors:
            memory.completed_colors.append(memory.held_color)
        try:
            rx, ry = _robot_xy(observation.robot_status)
            if memory.held_color and not _too_close_to_a(rx, ry):
                _store_pad_memory(memory.held_color, success_xy=(float(rx), float(ry)), approach_xy=(float(rx), float(ry)))
        except Exception:
            pass
        memory.held_color = None
        memory.active_color = None
        memory.stage = "need_cube"

    elif action == "skip_target":
        memory.stage = "need_cube"

    elif action in ("search_cube", "navigate_to_cube") and memory.held_color is None:
        memory.stage = "need_cube"
        # 속도전: navigate_to_cube 에서만 active_color 고정 (실제 visible cube 타겟).
        # search_cube(큐브 안 보임)일 때는 active_color 를 비워 특정 색 고착 방지.
        if action == "navigate_to_cube" and color:
            memory.active_color = color
        else:
            memory.active_color = None
    elif action in ("search_pad", "navigate_to_pad") and memory.held_color is not None:
        memory.stage = "need_pad"

    # navigate 통합 auto-pick/auto-place 전환 보정
    if action == "navigate_to_cube" and verified.get("picked_ok") and memory.held_color is not None:
        memory.active_color = memory.held_color
        memory.stage = "need_pad"
        memory.failed_attempts.pop(memory.held_color, None)
        try:
            rx, ry = _robot_xy(observation.robot_status)
            _A_POSITION = (float(rx), float(ry))
        except Exception:
            pass
    if action == "navigate_to_pad" and verified.get("placed_ok"):
        if memory.held_color and memory.held_color not in memory.completed_colors:
            memory.completed_colors.append(memory.held_color)
        try:
            rx, ry = _robot_xy(observation.robot_status)
            if memory.held_color and not _too_close_to_a(rx, ry):
                _store_pad_memory(memory.held_color, success_xy=(float(rx), float(ry)), approach_xy=(float(rx), float(ry)))
        except Exception:
            pass
        memory.held_color = None
        memory.active_color = None
        memory.stage = "need_cube"

    # stage/held 일관성 보정
    if memory.held_color is not None and memory.stage == "need_cube":
        memory.stage = "need_pad"
    if memory.held_color is None and memory.stage == "need_pad":
        memory.stage = "need_cube"

    if action == "pick_cube" and not verified.get("picked_ok") and color:
        memory.failed_attempts[color] = memory.failed_attempts.get(color, 0) + 1
    if action == "place_cube" and not verified.get("placed_ok") and memory.held_color:
        c = memory.held_color
        memory.failed_attempts[c] = memory.failed_attempts.get(c, 0) + 1
        memory.failed_attempts[f"{c}:place"] = memory.failed_attempts.get(f"{c}:place", 0) + 1
    if action == "pick_cube" and not verified.get("picked_ok") and color:
        memory.failed_attempts[f"{color}:pick"] = memory.failed_attempts.get(f"{color}:pick", 0) + 1

    _sync_memory_views(memory)
    memory.logs.append({
        "stage": memory.stage, "held_color": memory.held_color,
        "delivered_count": memory.delivered_count,
        "completed_colors": list(memory.completed_colors),
        "failed_attempts": dict(memory.failed_attempts),
        "decision": decision.__dict__,
        "verified_summary": {k: v for k, v in verified.items() if k in ("picked_ok", "placed_ok", "reached")},
        "visible_count": len(observation.detections),
        "pad_positions": dict(_PAD_POSITIONS),
        "pad_approach_poses": dict(_PAD_APPROACH_POSES),
        "place_success_poses": dict(_PLACE_SUCCESS_POSES),
        "last_sign_bearing": dict(_LAST_SIGN_BEARING),
        "color_memory": memory.color_memory,
    })


async def recover_motion(ctx: Any, memory: AgentMemory, reason: str | None = None) -> dict[str, Any]:
    """후퇴 + 회전 + 재스캔. 같은 색 3회 초과 실패 시 skip 유도.
    전도 상태거나 RPC timeout 시 act를 건너뛰어 런 전체 크래시 방지."""
    color = memory.active_color or memory.held_color
    if color and memory.failed_attempts.get(color, 0) >= 3:
        if color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        return {"action": "recover", "reason": reason, "status": "suggest_skip", "color": color}

    async def _safe(aw):
        try:
            return await aw
        except Exception as exc:
            print(f"[recover] step skipped: {type(exc).__name__}: {exc}")
            return None

    await _safe(move_velocity(ctx, vx=-0.15, duration_s=0.8))
    await _safe(asyncio.sleep(0.2))
    await _safe(move_velocity(ctx, wz=0.4, duration_s=1.0))
    await _safe(asyncio.sleep(0.2))
    try:
        await scan_head(ctx, yaws=(-0.8, 0.0, 0.8), pitch=0.15)
    except Exception:
        pass
    return {"action": "recover", "reason": reason, "status": "stepped_back_and_rotated"}


# ---------------------------------------------------------------------------
# LEVEL 1 학생 구현: A 컨베이어 유도 + coordinate-guided action
# ---------------------------------------------------------------------------
async def navigate_toward_conveyor_a(ctx: Any, memory: AgentMemory) -> dict[str, Any]:
    """need_cube + 큐브 미가시 시: VLM sign A 방향으로 본체 회전 후 전진.
    A xy를 추정해 go_to 한 번 진행하고 재스캔을 유도한다."""
    global _A_POSITION
    await set_head(ctx, yaw=0.0, pitch=0.15)
    await asyncio.sleep(0.3)
    bearing = None
    try:
        raw = await ask_vlm_about_frame(ctx, build_conveyor_a_vlm_prompt(), api_key=_tokamak_api_key())
        bearing = _parse_a_bearing_from_text(raw)
    except Exception:
        bearing = None

    # bearing 없으면 일단 회전하며 탐색
    if bearing is None:
        await move_velocity(ctx, wz=0.35, duration_s=0.9)
        return {"action": "search_cube", "status": "a_not_found_turned", "turn": memory.search_turns}

    clipped = max(-VLM_GUIDE_MAX_TURN_DEG, min(VLM_GUIDE_MAX_TURN_DEG, float(bearing)))
    await safe_turn_to_bearing(ctx, clipped, tolerance_deg=12.0, max_turn_deg=VLM_GUIDE_MAX_TURN_DEG)
    await asyncio.sleep(0.15)

    # 기록된 A 좌표가 있으면 go_to로 접근 (직접 기록한 좌표 → 규칙 OK)
    if _A_POSITION is not None:
        try:
            await _go_to_xy(ctx, _A_POSITION[0], _A_POSITION[1])
        except Exception:
            pass
    else:
        # 좌표 미기록 시 전진 후 재스캔
        await move_velocity(ctx, vx=0.22, duration_s=0.8)
    return {"action": "search_cube", "status": "headed_toward_a", "a_bearing": bearing, "clipped_bearing": clipped}


# ---------------------------------------------------------------------------
# LEVEL 1 학생 구현: coordinate-guided action
# ---------------------------------------------------------------------------
def estimate_target_xy_from_observation(
    observation: Observation,
    target_color: str | None,
) -> tuple[float, float] | None:
    """Camera bearing + blob_area 거리 추정으로 target world x/y 계산 (큐브용)."""
    detections = observation.detections or []
    cand = None
    if target_color is None:
        if not detections:
            return None
        cand = max(detections, key=lambda d: d.blob_area)
    else:
        # 큐브만: pad/signage blob 제외
        same = _filter_cubes([d for d in detections if d.color == target_color])
        if not same:
            return None
        cand = same[0]

    if cand.blob_area < 80:
        return None

    rx, ry = _robot_xy(observation.robot_status)
    yaw = _robot_yaw_rad(observation.robot_status)
    bearing_body = math.radians(getattr(cand, "full_bearing_deg", cand.angle_deg))
    bearing_deg = math.degrees(bearing_body)
    if abs(bearing_deg) > CUBE_NAV_MAX_BEARING_DEG:
        _log_jsonl({
            "event": "cube_nav_rejected",
            "reason": "bearing_too_wide",
            "color": getattr(cand, "color", target_color),
            "bearing_deg": round(bearing_deg, 2),
            "blob_area": int(getattr(cand, "blob_area", 0) or 0),
            "area_ratio": _area_ratio(cand),
        })
        return None
    world_bearing = yaw + bearing_body
    raw_distance = _distance_from_blob_area(cand.blob_area)
    if raw_distance > CUBE_NAV_MAX_RAW_DISTANCE_M:
        _log_jsonl({
            "event": "cube_nav_rejected",
            "reason": "raw_distance_too_far",
            "color": getattr(cand, "color", target_color),
            "raw_distance_m": round(raw_distance, 3),
            "blob_area": int(getattr(cand, "blob_area", 0) or 0),
            "area_ratio": _area_ratio(cand),
        })
        return None
    # 컨베이어/장애물에 충돌·전도 방지: 큐브 추정 위치 0.35m 전에서 멈춤.
    distance = min(max(raw_distance - 0.35, 0.3), CUBE_NAV_MAX_STEP_M)
    tx = rx + distance * math.cos(world_bearing)
    ty = ry + distance * math.sin(world_bearing)
    if not _point_in_safe_world(tx, ty):
        _log_jsonl({
            "event": "cube_nav_rejected",
            "reason": "target_out_of_world_bounds",
            "color": getattr(cand, "color", target_color),
            "target_xy": [round(float(tx), 3), round(float(ty), 3)],
            "raw_distance_m": round(raw_distance, 3),
            "step_m": round(distance, 3),
        })
        return None
    return (float(tx), float(ty))


def estimate_pad_xy_from_observation(
    observation: Observation,
    held_color: str,
) -> tuple[float, float] | None:
    """패드 destination sign(큰 blob)만으로 xy 추정. 큐브/conveyor blob 사용 금지."""
    pads = _filter_pads(observation.detections, held_color)
    if not pads:
        return None
    cand = pads[0]
    rx, ry = _robot_xy(observation.robot_status)
    yaw = _robot_yaw_rad(observation.robot_status)
    bearing_body = math.radians(getattr(cand, "full_bearing_deg", cand.angle_deg))
    world_bearing = yaw + bearing_body
    raw_distance = _distance_from_pad_blob_area(cand.blob_area)
    # 패드 앞 0.4m에서 멈춤
    distance = max(raw_distance - 0.4, 0.6)
    tx = rx + distance * math.cos(world_bearing)
    ty = ry + distance * math.sin(world_bearing)
    if _too_close_to_a(tx, ty):
        return None
    return (float(tx), float(ty))


def estimate_pad_center_xy_from_observation(
    observation: Observation,
    held_color: str,
) -> tuple[tuple[float, float], Any] | None:
    """패드 중심 후보와 사용한 detection을 반환한다. 접근 pose는 별도 함수에서 계산."""
    pads = _filter_pads_safe(observation.detections, held_color, observation.robot_status)
    if not pads:
        return None
    cand = pads[0]
    raw_distance = _distance_from_pad_blob_area(int(getattr(cand, "blob_area", 0) or 0))
    distance = max(raw_distance, 0.8)
    center = _world_xy_from_detection(observation.robot_status, cand, distance)
    if _too_close_to_a(center[0], center[1]):
        return None
    return center, cand


def estimate_pad_approach_from_observation(
    observation: Observation,
    held_color: str,
    *,
    retry: bool = False,
) -> tuple[tuple[float, float], tuple[float, float], Any] | None:
    """관찰 기반 pad 중심을 찾고, 반사각/좌측 offset 접근 pose를 만든다."""
    center_and_det = estimate_pad_center_xy_from_observation(observation, held_color)
    if center_and_det is None:
        return None
    center, det = center_and_det
    robot_xy = _robot_xy(observation.robot_status)
    approach = _safe_pad_approach_pose(robot_xy, center, retry=retry)
    if _too_close_to_a(approach[0], approach[1], margin=1.1):
        return None
    return approach, center, det


async def _visual_creep_to_pad(
    ctx: Any,
    held_color: str,
    *,
    target_area: int = 45000,
    max_steps: int = 4,
) -> bool:
    """Pick의 close-in 접근과 동일한 원리: 한 번의 거리 공식 대신
    스캔→회전(중앙 정렬)→전진을 반복해 실제로 패드 앞까지 다가간다.
    One-shot bearing+blob면적 공식은 원거리에서 오차가 커서 패드를 벗어나는 문제를 해결한다."""
    found_once = False
    effective_steps = max_steps
    if held_color == "green" and not _is_pad_sign_confirmed(held_color):
        effective_steps = min(max_steps, 2)
    for _ in range(effective_steps):
        status = await get_robot_status(ctx)
        rx, ry = _robot_xy(status)
        if _too_close_to_a(rx, ry, margin=0.9):
            await move_velocity(ctx, vx=0.3, duration_s=1.0)
            continue
        dets = await scan_head(ctx, yaws=PAD_CREEP_SCAN_YAWS, pitch=0.15)
        pads = _filter_pads_safe(dets, held_color, status)
        if not pads:
            if held_color == "green":
                return found_once
            await move_velocity(ctx, wz=0.45, duration_s=0.5)
            continue
        found_once = True
        best = pads[0]
        bearing_deg = getattr(best, "full_bearing_deg", best.angle_deg)
        if await safe_turn_to_bearing(ctx, float(bearing_deg), tolerance_deg=8.0):
            continue
        if held_color == "green" and not _is_pad_sign_confirmed(held_color):
            confirmed = await _confirm_destination_sign_quick(ctx, held_color, yaws=(0.0,))
            if not confirmed:
                _log_jsonl({
                    "event": "pad_creep_rejected",
                    "held_color": held_color,
                    "reason": "green C sign not confirmed",
                    "target_bearing": round(float(bearing_deg), 2),
                })
                return False
        ratio = _area_ratio(best)
        if held_color == "green":
            _store_pad_memory(
                held_color,
                sign_bearing=float(bearing_deg),
            )
        else:
            _LAST_SIGN_BEARING[held_color] = float(bearing_deg)
        _log_jsonl({
            "event": "pad_creep",
            "held_color": held_color,
            "target_bearing": round(float(bearing_deg), 2),
            "blob_area": int(getattr(best, "blob_area", 0) or 0),
            "area_ratio": round(ratio, 5),
        })
        if best.blob_area >= target_area or ratio >= PAD_PLACE_AREA_RATIO:
            return True
        await safe_nudge_forward(ctx, vx=0.35, duration_s=0.9)
    return found_once


async def _search_pad_xy_by_rotation(
    ctx: Any,
    held_color: str,
    *,
    max_turns: int = 6,
) -> tuple[float, float] | None:
    """들고 있는 색의 destination pad를 본체 회전 스캔으로 찾고 반사각 접근 pose를 반환한다."""
    best_xy: tuple[float, float] | None = None
    best_rank = 0.0
    effective_turns = min(max_turns, 3) if held_color == "green" and not _is_pad_sign_confirmed(held_color) else max_turns
    for _ in range(effective_turns):
        status = await get_robot_status(ctx)
        dets = await scan_head(ctx, yaws=(-0.8, -0.4, 0.0, 0.4, 0.8), pitch=0.15)
        for pad in _filter_pads_safe(dets, held_color, status):
            if held_color == "green" and not _is_pad_sign_confirmed(held_color):
                if not await _confirm_destination_sign_quick(ctx, held_color, yaws=(0.0,)):
                    _log_jsonl({
                        "event": "pad_candidate_rejected",
                        "held_color": held_color,
                        "reason": "green C sign not confirmed",
                        "blob_area": int(getattr(pad, "blob_area", 0) or 0),
                        "area_ratio": round(_area_ratio(pad), 5),
                    })
                    continue
            obs = Observation(robot_status=status, detections=[pad])
            estimated = estimate_pad_approach_from_observation(obs, held_color)
            if estimated is None:
                continue
            approach, center, det = estimated
            rank = max(_area_ratio(pad), float(getattr(pad, "blob_area", 0) or 0) / 100000.0)
            if rank > best_rank:
                best_rank = rank
                best_xy = approach
                _store_pad_memory(
                    held_color,
                    center_xy=center,
                    approach_xy=approach,
                    sign_bearing=float(getattr(det, "full_bearing_deg", getattr(det, "angle_deg", 0.0)) or 0.0),
                )
        if best_xy is not None:
            return best_xy
        await move_velocity(ctx, wz=0.55, duration_s=1.2)
        await asyncio.sleep(0.15)
    return best_xy


async def _go_to_deliver_xy(ctx: Any, held_color: str, x: float, y: float) -> Any:
    """배달용 go_to — held 색과 다른 패드 앵커(E/C 혼동)면 거부."""
    if not _deliver_xy_matches_held_color(held_color, float(x), float(y)):
        nearest = _nearest_pad_color(float(x), float(y))
        _log_jsonl({
            "event": "go_to_rejected",
            "reason": "deliver_target_wrong_pad",
            "held_color": held_color,
            "expected_sign": DESTINATION_SIGN_RULES.get(held_color),
            "nearest_pad_color": nearest,
            "target_xy": [round(float(x), 3), round(float(y), 3)],
            "recorded_pad_xy": list(_PAD_POSITIONS.get(held_color, (0.0, 0.0))),
        })
        raise RuntimeError(
            f"deliver go_to ({x:.2f}, {y:.2f}) matches pad {nearest}, not {held_color}"
        )
    return await _go_to_xy(ctx, x, y)


async def _go_to_xy(ctx: Any, x: float, y: float) -> Any:
    """Coordinate-based go_to입니다. 학생 시스템이 추정/기록한 x/y에만 사용하세요."""
    if not _point_in_safe_world(float(x), float(y)):
        _log_jsonl({
            "event": "go_to_rejected",
            "reason": "target_out_of_world_bounds",
            "target_xy": [round(float(x), 3), round(float(y), 3)],
            "bounds": list(SAFE_WORLD_BOUNDS),
        })
        raise RuntimeError(f"go_to target outside safe world bounds: ({x:.2f}, {y:.2f})")
    status = await get_robot_status(ctx)
    start_xy = _robot_xy(status)
    route = _plan_safe_route(start_xy, (float(x), float(y)))
    waypoints = route["waypoints"]
    _log_jsonl({
        "event": "shelf_route",
        "shelf_avoidance_applied": bool(route["shelf_avoidance_applied"]),
        "blocked_rect": route["blocked_rect"],
        "route_waypoints": [[round(px, 3), round(py, 3)] for px, py in waypoints],
        "route_distance_m": route["route_distance_m"],
        "route_source": route["route_source"],
        "target_xy": [round(float(x), 3), round(float(y), 3)],
        "original_target": [round(route["original_target"][0], 3), round(route["original_target"][1], 3)],
        "target_adjusted": bool(route["target_adjusted"]),
    })
    if not waypoints:
        raise RuntimeError(f"no safe shelf route to ({x:.2f}, {y:.2f})")

    result: Any = None
    for idx, (wx, wy) in enumerate(waypoints, start=1):
        t0 = time.perf_counter()
        try:
            result = await ctx.invoke(
                "go_to",
                {
                    "target": {
                        "kind": "pose",
                        "pose": {"frame_id": "world", "position": [float(wx), float(wy), 0]},
                    }
                },
                timeout_s=90,
            )
        finally:
            elapsed = time.perf_counter() - t0
            _log_jsonl({"event": "skill_latency", "skill": "go_to", "seconds": round(elapsed, 4)})
        _log_jsonl({
            "event": "shelf_route_waypoint",
            "index": idx,
            "count": len(waypoints),
            "xy": [round(float(wx), 3), round(float(wy), 3)],
            "route_source": route["route_source"],
            "shelf_avoidance_applied": bool(route["shelf_avoidance_applied"]),
            "status": "done",
            "result": result_summary(result),
        })
    return result


async def go_to_xy(ctx: Any, x: float, y: float) -> Any:
    """Coordinate-based go_to입니다. 학생 시스템이 추정한 x/y에만 사용하세요."""
    return await _go_to_xy(ctx, x, y)


async def _close_in_and_pick(ctx: Any, pick_threshold: int = 8000) -> dict[str, Any] | None:
    """go_to 후 잔여 거리를 완만한 시각 접근으로 줄여 즉시 pick.
    정면 스캔 → 큐브가 충분히 크면 pick; 아니면 전진(0.2m/s·1.5s=0.3m) 후 재스캔·pick.
    전도 방지: area < 7000(너무 멂)이면 pick 않고 None 반환(다음 cycle 재접근 유도).
    pick 성공 시 result_summary 반환, 실패 시 None."""
    return await _close_in_and_pick_color(ctx, target_color=None, pick_threshold=pick_threshold)


async def _close_in_and_pick_color(
    ctx: Any,
    *,
    target_color: str | None,
    pick_threshold: int = 8000,
) -> dict[str, Any] | None:
    """target_color 지정 시 해당 색만, None이면 TARGET_COLORS 전체를 허용."""
    allowed = {target_color} if target_color else set(TARGET_COLORS)
    try:
        await set_head(ctx, yaw=0.0, pitch=0.15)
    except Exception:
        pass
    for _ in range(4):
        dets = await scan_head(ctx, yaws=(-0.35, 0.0, 0.35), pitch=0.15)
        cubes = [c for c in _filter_cubes(dets) if c.color in allowed]
        target = _pick_preferred_cube(cubes)
        if target and (target.blob_area >= pick_threshold or _area_ratio(target) >= CUBE_PICK_AREA_RATIO):
            bearing = getattr(target, "full_bearing_deg", target.angle_deg)
            if await safe_turn_to_bearing(ctx, float(bearing), tolerance_deg=18.0):
                continue
            return result_summary(await pick_nearest_cube(ctx))
        if target:
            await safe_nudge_forward(ctx, vx=0.2, duration_s=1.0)
        else:
            await move_velocity(ctx, wz=0.25, duration_s=0.5)
    dets2 = await scan_head(ctx, yaws=(0.0,), pitch=0.15)
    cubes2 = [c for c in _filter_cubes(dets2) if c.color in allowed]
    if cubes2 and (cubes2[0].blob_area >= 7000 or _area_ratio(cubes2[0]) >= CUBE_PICK_AREA_RATIO):
        return result_summary(await pick_nearest_cube(ctx))
    return None


async def _close_in_and_place(ctx: Any, held_color: str | None, *, trust_recorded: bool = False) -> dict[str, Any] | None:
    """go_to pad 후 place. recorded pad 경로(trust_recorded)는 VLM 생략, A 근처 place만 차단."""
    if not held_color:
        return None
    try:
        status = await get_robot_status(ctx)
        rx, ry = _robot_xy(status)
    except Exception:
        rx, ry = 0.0, 0.0
    if _too_close_to_a(rx, ry):
        return None
    if held_color == "green" and not _is_pad_sign_confirmed(held_color):
        if not await _vlm_confirms_destination(ctx, held_color):
            return None
    elif not trust_recorded and not await _vlm_confirms_destination(ctx, held_color):
        return None
    res = await place_nearest_zone(ctx)
    summary = result_summary(res)
    if summary.get("status") == "done":
        return summary
    await move_velocity(ctx, vx=0.15, duration_s=0.8)
    if held_color == "green" and not _is_pad_sign_confirmed(held_color):
        if not await _vlm_confirms_destination(ctx, held_color):
            return None
    elif not trust_recorded and not await _vlm_confirms_destination(ctx, held_color):
        return None
    return result_summary(await place_nearest_zone(ctx))


# ---------------------------------------------------------------------------
# LEVEL 1 속도전: pad 스윕 + 결정적 pick/deliver
# ---------------------------------------------------------------------------
def _preview_next_target(detections: list[Any], memory: AgentMemory) -> None:
    """컨베이어에 보이는 4색 큐브 중 다음 pick 색을 memory.active_color에 기록."""
    if memory.held_color is not None:
        return
    if not _looks_like_conveyor_source(detections):
        return
    cubes = _candidate_pick_cubes(detections)
    if not cubes:
        return
    if memory.active_color in TARGET_COLORS:
        preferred = [c for c in cubes if c.color == memory.active_color]
        if preferred:
            return
    closest = _pick_preferred_cube(cubes) or cubes[0]
    memory.active_color = closest.color


def _candidate_pick_cubes(detections: list[Any]) -> list[Any]:
    """현재 frame에서 pick 대상으로 볼 수 있는 큐브 후보 (가까운/큰 blob 우선)."""
    cands = [
        c for c in _filter_cubes(detections)
        if c.color in TARGET_COLORS and (c.blob_area >= 1200 or _area_ratio(c) >= 0.003)
    ]

    def rank(det: Any) -> float:
        bearing = abs(float(getattr(det, "full_bearing_deg", getattr(det, "angle_deg", 0.0)) or 0.0))
        area = min(float(getattr(det, "blob_area", 0) or 0), 14000.0)
        return bearing - (area / 14000.0 * 10.0)

    cands.sort(key=rank)
    return cands


def _is_visible_pick_candidate(det: Any) -> bool:
    """눈앞에서 짧은 보정만으로 집을 만한 cube 후보인지 판정한다."""
    bearing = abs(float(getattr(det, "full_bearing_deg", getattr(det, "angle_deg", 0.0)) or 0.0))
    if bearing > PICK_VISIBLE_MAX_BEARING_DEG:
        return False
    area = int(getattr(det, "blob_area", 0) or 0)
    return area >= PICK_VISIBLE_MIN_AREA and _area_ratio(det) >= PICK_VISIBLE_MIN_AREA_RATIO


def _visible_pick_candidates(detections: list[Any]) -> list[Any]:
    return [c for c in _candidate_pick_cubes(detections) if _is_visible_pick_candidate(c)]


def _looks_like_conveyor_source(detections: list[Any]) -> bool:
    """A source 앞에서 보이는 여러 색 큐브 묶음인지 판정한다.

    단일 작은 blob만으로 A를 기록하면 선반/표지판을 큐브로 오인할 수 있으므로,
    source belt처럼 여러 색의 cube-like blob이 동시에 보일 때만 fallback으로 쓴다.
    """
    cubes = [
        c
        for c in _filter_cubes(detections)
        if c.color in TARGET_COLORS
        and (
            900 <= int(getattr(c, "blob_area", 0) or 0) <= 10000
            or 0.002 <= _area_ratio(c) <= 0.035
        )
        and abs(float(getattr(c, "full_bearing_deg", getattr(c, "angle_deg", 0.0)) or 0.0)) <= 60.0
    ]
    if not any(_is_visible_pick_candidate(c) for c in cubes):
        return False
    colors = {c.color for c in cubes}
    if len(cubes) < 3:
        return False
    if len(colors) >= 2:
        return True
    # belt에 한 색만 노출돼도 blob이 여러 개면 source로 본다 (setup key별 단색 구간).
    return len(cubes) >= 4


def _senses_conveyor_source(detections: list[Any]) -> bool:
    """A source 벨트처럼 여러 색 큐브 묶음이 가까이 있는지 센싱 판정.

    _is_cube_like는 인접 큐브가 하나로 합쳐진 큰 blob을 거르므로 source 탐지에 부적합.
    여기선 색 blob 크기/위치만으로 multi-color 군집을 인식한다(개별 큐브 분리 불필요).
    pad 같은 거대 바닥 blob은 형상 비율로 제외한다.
    """
    blobs: list[Any] = []
    for d in detections:
        color = getattr(d, "color", None)
        if color not in TARGET_COLORS:
            continue
        area = int(getattr(d, "blob_area", 0) or 0)
        ratio = _area_ratio(d)
        if area < 400 and ratio <= 0.0:
            continue
        bearing = abs(float(getattr(d, "full_bearing_deg", getattr(d, "angle_deg", 0.0)) or 0.0))
        if bearing > 70.0:
            continue
        if _centroid_y_ratio(d) < 0.22:
            continue
        w, h = _bbox_wh(d)
        if w > 0 and h > 0:
            image_size = getattr(d, "image_size", (0, 0))
            try:
                iw, ih = int(image_size[0]), int(image_size[1])
            except Exception:
                iw, ih = 0, 0
            if iw and ih and (w / float(iw) > 0.60 or h / float(ih) > 0.55):
                continue
        blobs.append(d)
    if not blobs:
        return False
    colors = {getattr(d, "color", None) for d in blobs}
    if len(colors) >= 2 and len(blobs) >= 2:
        return True
    return len(blobs) >= 3


VLM_BODY_SCAN_STEPS = 6
VLM_BODY_SCAN_WZ = 0.5
# 60°(≈1.047 rad) / 0.5 rad/s ≈ 2.1s
VLM_BODY_SCAN_STEP_DURATION_S = 2.1


async def _vlm_scan_for_a_and_cubes(ctx: Any, memory: AgentMemory) -> dict[str, Any]:
    """몸통을 60° 간격으로 360° 회전하며 VLM+color scan으로 A/큐브를 찾는다.

    사용자 요청: "각도 틀면서 사진 다 찍고 어떤 거 할지 생각해야 됨 — 먼저 찍고 출발할지 pickup할지".
    각 정지에서 scan_head + VLM A 표지판 질문을 병행하고 세 가지로 분류한다:
      - pick_now:  눈앞에 집을 만한 색 큐브가 보임 → 먼저 pick
      - navigate_a: A 컨베이어/표지판이 보임 → 해당 bearing로 출발
      - nothing:  없으면 다음 각도로 계속
    """
    await set_head(ctx, yaw=0.0, pitch=0.15)
    best: dict[str, Any] = {"decision": "nothing", "vlm_bearing": None, "cube_color": None,
                            "source_like": False, "a_visible": False, "step": 0}

    for step in range(1, VLM_BODY_SCAN_STEPS + 1):
        try:
            dets = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
        except Exception:
            dets = []
        _preview_next_target(dets, memory)

        vlm_bearing: float | None = None
        a_visible = False
        try:
            vlm_raw = await ask_vlm_about_frame(ctx, build_conveyor_a_vlm_prompt(), api_key=_tokamak_api_key())
            a_visible = bool(vlm_raw) and "a_visible" in (vlm_raw or "").lower() and not (
                "false" in (vlm_raw or "").lower().split('"a_visible"', 1)[1][:30]
            )
            vlm_bearing = _parse_a_bearing_from_text(vlm_raw)
        except Exception as exc:
            _log_jsonl({
                "event": "vlm_body_scan_vlm_error",
                "step": step,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })

        source_like = _looks_like_conveyor_source(dets)
        pick_cands = _visible_pick_candidates(dets)
        cube_color = pick_cands[0].color if pick_cands else None

        decision = "nothing"
        if pick_cands:
            decision = "pick_now"
        elif source_like or a_visible:
            decision = "navigate_a"

        robot_xy = None
        try:
            robot_xy = list(_robot_xy(await get_robot_status(ctx)))
        except Exception:
            pass

        _log_jsonl({
            "event": "vlm_body_scan",
            "step": step,
            "decision": decision,
            "a_visible": a_visible,
            "source_like": source_like,
            "vlm_bearing": vlm_bearing,
            "cube_color": cube_color,
            "pick_candidates": len(pick_cands),
            "robot_xy": robot_xy,
        })

        if decision != "nothing":
            best = {"decision": decision, "vlm_bearing": vlm_bearing, "cube_color": cube_color,
                    "source_like": source_like, "a_visible": a_visible, "step": step}
            return best

        # 다음 60° 로 회전
        try:
            await move_velocity(ctx, wz=VLM_BODY_SCAN_WZ, duration_s=VLM_BODY_SCAN_STEP_DURATION_S)
        except Exception:
            pass

    return best


async def _ensure_at_conveyor_a(ctx: Any, memory: AgentMemory) -> bool:
    """랜덤 spawn 후 센싱/VLM으로 컨베이어 A를 찾아 다가가고 _A_POSITION 기록.

    흐름 (정답 좌표 없이 센싱만 사용):
      0) 현재 위치 1회 스캔 — spawn이 컨베이어 근처면 바로 pick/기록.
      1) 제자리 360° 회전 센싱 스캔 — multi-color source blob 또는 VLM A 표지판(1회 보조).
         source 군집이 보이면 그 방향으로 전진하며 좁히고 A 기록.
      2) 동쪽 소폭 전진 후 재스캔.
      3) 최후 fallback: VLM 1회 가이드 + navigate_toward_conveyor_a.
    """
    global _A_POSITION

    async def _record_a_here(*, reason: str) -> bool:
        global _A_POSITION
        try:
            status = await get_robot_status(ctx)
            _A_POSITION = _robot_xy(status)
            memory.recent_a_position = _A_POSITION
            _log_jsonl({
                "event": "a_position_recorded",
                "reason": reason,
                "xy": [round(_A_POSITION[0], 3), round(_A_POSITION[1], 3)],
            })
            return True
        except Exception as exc:
            _log_jsonl({
                "event": "a_position_record_failed",
                "reason": reason,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            return False

    if _A_POSITION is not None:
        try:
            status = await get_robot_status(ctx)
            rx, ry = _robot_xy(status)
            if _dist_xy((rx, ry), _A_POSITION) <= SOURCE_RING_RADIUS_M:
                dets = await scan_head(ctx, yaws=(-0.4, 0.0, 0.4), pitch=0.15)
                _preview_next_target(dets, memory)
                if _looks_like_conveyor_source(dets) or memory.active_color in TARGET_COLORS:
                    return True
            await _go_to_xy(ctx, _A_POSITION[0], _A_POSITION[1])
            return True
        except Exception:
            pass

    # 0) 현재 위치 1회 스캔 — spawn이 컨베이어 근처면 바로 pick/기록.
    try:
        dets0 = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
        _preview_next_target(dets0, memory)
        if _visible_pick_candidates(dets0):
            _log_jsonl({"event": "a_scan_decision", "decision": "pick_now",
                        "reason": "spawn_visible_pick_candidates"})
            return True
        if _senses_conveyor_source(dets0):
            await _record_a_here(reason="spawn_source_like")
            return True
    except Exception as exc:
        _log_jsonl({"event": "a_spawn_scan_error", "error_type": type(exc).__name__, "error": str(exc)})

    # 1) 제자리 360° 회전 센싱 스캔 — multi-color source blob 또는 VLM A 표지판(1회 보조).
    #    정답 좌표 없이 센싱으로 A 방향을 잡고, source 군집이 보이면 그 방향으로 전진해 좁힌다.
    for step in range(SENSING_A_SCAN_STEPS):
        try:
            dets = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
        except Exception:
            dets = []
        _preview_next_target(dets, memory)
        if _visible_pick_candidates(dets):
            _log_jsonl({"event": "a_scan_decision", "decision": "pick_now",
                        "reason": "sensing_visible_pick_candidates", "step": step})
            return True
        if _senses_conveyor_source(dets):
            await _record_a_here(reason=f"sensing_source_like_step{step}")
            return True

        # VLM A-sign 보조는 첫 step에서 1회만.
        if step == 0:
            bearing = None
            try:
                await set_head(ctx, yaw=0.0, pitch=0.15)
                vlm_raw = await ask_vlm_about_frame(
                    ctx, build_conveyor_a_vlm_prompt(), api_key=_tokamak_api_key())
                bearing = _parse_a_bearing_from_text(vlm_raw)
            except Exception as exc:
                _log_jsonl({"event": "a_search_vlm_error", "step": step,
                            "error_type": type(exc).__name__, "error": str(exc)})
            if bearing is not None:
                clipped = max(-VLM_GUIDE_MAX_TURN_DEG, min(VLM_GUIDE_MAX_TURN_DEG, float(bearing)))
                _log_jsonl({"event": "a_search_vlm_guide", "step": step,
                            "vlm_bearing": bearing, "clipped_bearing": clipped})
                if abs(clipped) > 15:
                    await safe_turn_to_bearing(ctx, clipped, tolerance_deg=15.0,
                                               max_turn_deg=VLM_GUIDE_MAX_TURN_DEG)
                else:
                    await move_velocity(ctx, vx=0.18, duration_s=0.6)
                dets_after = await scan_head(ctx, yaws=(-0.5, 0.0, 0.5), pitch=0.15)
                if _senses_conveyor_source(dets_after):
                    await _record_a_here(reason="sensing_vlm_a_source_like")
                    return True
                if _visible_pick_candidates(dets_after):
                    return True
                # VLM이 가리킨 방향으로 전진하며 source를 좁힌다.
                await move_velocity(ctx, vx=0.22, duration_s=1.2)
                continue

        # 회전하며 탐색
        try:
            await move_velocity(ctx, wz=SENSING_A_SCAN_WZ,
                                duration_s=SENSING_A_SCAN_STEP_DURATION_S)
        except Exception:
            pass

    # 2) 동쪽(맵 중심)으로 소폭 전진 후 재스캔 — 회전만으로 못 잡았을 때.
    try:
        await move_velocity(ctx, vx=0.25, duration_s=1.5)
    except Exception:
        pass
    dets = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
    if _visible_pick_candidates(dets):
        return True
    if _senses_conveyor_source(dets):
        await _record_a_here(reason="sensing_east_probe_source_like")
        return True

    # 3) 최후 fallback: VLM 1회 가이드 + navigate_toward_conveyor_a
    await navigate_toward_conveyor_a(ctx, memory)
    dets = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
    if _senses_conveyor_source(dets):
        _log_jsonl({"event": "a_source_candidate", "reason": "post_navigate_source_like"})
        return True
    return _A_POSITION is not None


async def calibrate_pads(ctx: Any, *, budget_s: float = 90.0) -> dict[str, tuple[float, float]]:
    """시작 1회: A 찾기 → B/C/D/E 패드 스윕·기록 → A 복귀."""
    global _A_POSITION, _PAD_POSITIONS
    t0 = time.perf_counter()
    memory_stub = AgentMemory()

    for stale in list(_PAD_POSITIONS.keys()):
        if stale not in TARGET_COLORS:
            _PAD_POSITIONS.pop(stale, None)

    print("[calibrate] Finding conveyor A (random spawn)...")
    await _ensure_at_conveyor_a(ctx, memory_stub)
    if _A_POSITION:
        print(f"[calibrate] A position recorded: {_A_POSITION}")
    else:
        try:
            status = await get_robot_status(ctx)
            _A_POSITION = _robot_xy(status)
            print(f"[calibrate] A fallback (current pose): {_A_POSITION}")
        except Exception as exc:
            print(f"[calibrate] A not found: {exc}")

    best_pads: dict[str, Any] = {}
    for step in range(4):
        if time.perf_counter() - t0 > budget_s * 0.5:
            break
        status = await get_robot_status(ctx)
        dets = await scan_head(ctx, yaws=(-1.0, -0.5, 0.0, 0.5, 1.0), pitch=0.15)
        for pad in _filter_pads(dets):
            if pad.color not in TARGET_COLORS:
                continue
            prev = best_pads.get(pad.color)
            if prev is None or pad.blob_area > prev.blob_area:
                best_pads[pad.color] = pad
            obs = Observation(robot_status=status, detections=[pad])
            xy = estimate_pad_xy_from_observation(obs, pad.color)
            if xy and _A_POSITION and _dist_xy(xy, _A_POSITION) > 1.0:
                approach = _safe_pad_approach_pose(_robot_xy(status), xy)
                _store_pad_memory(pad.color, center_xy=xy, approach_xy=approach)
        await move_velocity(ctx, wz=0.5, duration_s=1.2)
        print(f"[calibrate] sweep {step + 1} pads={list(_PAD_POSITIONS.keys())}")

    refine_deadline = t0 + budget_s * 0.85
    skip_refine = budget_s <= 30.0
    if skip_refine:
        print("[calibrate] skip pad refine (short budget)")
    for color in list(_PAD_POSITIONS.keys()):
        if skip_refine:
            break
        if color not in TARGET_COLORS:
            continue
        if time.perf_counter() > refine_deadline:
            break
        xy = _PAD_POSITIONS.get(color)
        if not xy:
            continue
        try:
            await _go_to_xy(ctx, xy[0], xy[1])
            reached = await _visual_creep_to_pad(ctx, color, max_steps=3)
            status = await get_robot_status(ctx)
            rx, ry = _robot_xy(status)
            if reached and _A_POSITION and _dist_xy((rx, ry), _A_POSITION) > 1.0:
                _store_pad_memory(color, success_xy=(float(rx), float(ry)), approach_xy=(float(rx), float(ry)))
                sign = DESTINATION_SIGN_RULES.get(color, "?")
                print(f"[calibrate] refined {color}→{sign} at ({rx:.2f},{ry:.2f})")
                _log_jsonl({"event": "calibrate", "color": color, "sign": sign,
                            "xy": [rx, ry], "refined": True, "reached": True})
            elif not reached:
                print(f"[calibrate] refine {color} did not reach close range, keeping rough estimate")
                _log_jsonl({"event": "calibrate", "color": color, "refined": False, "reached": False})
        except Exception as exc:
            print(f"[calibrate] refine {color} skipped: {exc}")
            _log_jsonl({"event": "calibrate", "color": color, "error": str(exc)})

    if _A_POSITION and time.perf_counter() - t0 < budget_s:
        try:
            await _go_to_xy(ctx, _A_POSITION[0], _A_POSITION[1])
        except Exception as exc:
            print(f"[calibrate] return to A failed: {exc}")

    print(f"[calibrate] done in {time.perf_counter() - t0:.1f}s pads={dict(_PAD_POSITIONS)}")
    _log_jsonl({"event": "calibrate_done", "elapsed_s": round(time.perf_counter() - t0, 1),
                "pad_positions": dict(_PAD_POSITIONS)})
    return dict(_PAD_POSITIONS)


async def _return_to_a_if_recorded(ctx: Any) -> None:
    """pick 후 배달은 A 근처에서 출발하도록 기록된 A 좌표로 복귀."""
    if _A_POSITION is None:
        return
    try:
        status = await get_robot_status(ctx)
        if _dist_xy(_robot_xy(status), _A_POSITION) <= SOURCE_RING_RADIUS_M:
            _log_jsonl({"event": "return_to_a", "xy": list(_A_POSITION), "status": "already_near"})
            return
        await _go_to_xy(ctx, _A_POSITION[0], _A_POSITION[1])
        _log_jsonl({"event": "return_to_a", "xy": list(_A_POSITION), "status": "ok"})
    except Exception as exc:
        print(f"  return to A failed: {exc}")
        _log_jsonl({"event": "return_to_a", "xy": list(_A_POSITION), "status": "failed", "error": str(exc)})


async def _near_recorded_a(ctx: Any, *, radius_m: float = SOURCE_RING_RADIUS_M) -> bool:
    """기록된 A source 근처에 있을 때만 current camera cube 후보를 신뢰한다."""
    if _A_POSITION is None:
        return False
    try:
        status = await get_robot_status(ctx)
        return _dist_xy(_robot_xy(status), _A_POSITION) <= radius_m
    except Exception as exc:
        _log_jsonl({
            "event": "a_position_check_failed",
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        return False


async def _record_a_from_current_pose(ctx: Any, memory: AgentMemory, *, reason: str) -> bool:
    """A/source 접근 pose를 현재 robot pose로 기록한다."""
    global _A_POSITION
    try:
        status = await get_robot_status(ctx)
        _A_POSITION = _robot_xy(status)
        memory.recent_a_position = _A_POSITION
        _log_jsonl({
            "event": "a_position_recorded",
            "reason": reason,
            "xy": [round(_A_POSITION[0], 3), round(_A_POSITION[1], 3)],
        })
        return True
    except Exception as exc:
        _log_jsonl({
            "event": "a_position_record_failed",
            "reason": reason,
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        return False


async def _vlm_confirms_a_sign(ctx: Any) -> bool:
    """현재 POV에서 VLM이 A 컨베이어 표지판을 본다고 응답하는지 확인."""
    try:
        raw = await ask_vlm_about_frame(ctx, build_conveyor_a_vlm_prompt(), api_key=_tokamak_api_key())
    except Exception as exc:
        _log_jsonl({
            "event": "source_staging_vlm_error",
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        return False
    if not raw:
        return False
    low = raw.lower()
    if '"a_visible"' in low and 'false' in low.split('"a_visible"', 1)[1][:30]:
        return False
    return "a_visible" in low or "conveyor" in low or '"a"' in low


async def _try_pick_nearest_at_source(ctx: Any, memory: AgentMemory, *, source_label: str) -> dict[str, Any] | None:
    """A/source ring에서 카메라 후보가 약해도 가까운 cube를 한 번 집어본다."""
    global _A_POSITION
    try:
        result = await pick_nearest_cube(ctx)
    except Exception as exc:
        _log_jsonl({
            "event": "source_blind_pick",
            "source": source_label,
            "status": "error",
            "error_type": type(exc).__name__,
            "error": str(exc),
        })
        return None

    summary = result_summary(result)
    await _sync_held_from_scene(ctx, memory)
    if memory.held_color in TARGET_COLORS:
        memory.active_color = memory.held_color
        memory.stage = "need_pad"
        memory.failed_attempts["pick_missed"] = 0
        await _record_a_from_current_pose(ctx, memory, reason=f"{source_label}_pick_success")
        return {
            "action": "pick_at_a",
            "status": "ok",
            "held_color": memory.held_color,
            "source": source_label,
            "auto_pick": summary,
            "target_color": memory.held_color,
            "reason": "nearest cube picked at source staging; deliver by held_color",
        }

    _log_jsonl({
        "event": "source_blind_pick",
        "source": source_label,
        "status": "missed",
        "result": summary,
    })
    return None


async def _return_to_a_and_preview(ctx: Any, memory: AgentMemory) -> None:
    """A 복귀 후 컨베이어 4색 스캔으로 다음 active_color 기록."""
    await _return_to_a_if_recorded(ctx)
    try:
        dets = await scan_head(ctx, yaws=(-0.5, 0.0, 0.5), pitch=0.15)
        _preview_next_target(dets, memory)
    except Exception:
        pass


async def _sync_held_from_scene(ctx: Any, memory: AgentMemory) -> None:
    """held_cube_info 로 memory.held_color/stage 동기화. scene 타임아웃 시 기존 상태 유지."""
    try:
        held = await get_held_cube_info(ctx)
    except Exception:
        return
    if held:
        memory.held_color = held["color"]
        memory.stage = "need_pad" if held["color"] in TARGET_COLORS else "need_cube"
    elif memory.held_color is not None:
        memory.held_color = None
        memory.stage = "need_cube"


async def _pick_from_candidates(
    ctx: Any,
    memory: AgentMemory,
    target_cubes: list[Any],
    source_detections: list[Any],
    *,
    source_label: str,
) -> dict[str, Any]:
    """현재 scan에서 고른 큐브 후보를 향해 접근하고 nearest cube wrapper로 pick."""
    global _A_POSITION
    target = _pick_preferred_cube(target_cubes) or target_cubes[0]
    memory.active_color = target.color
    memory.failed_attempts["a_no_cube"] = 0

    target_bearing = float(getattr(target, "full_bearing_deg", getattr(target, "angle_deg", 0.0)) or 0.0)
    if abs(target_bearing) > PICK_VISIBLE_MAX_BEARING_DEG:
        return {
            "action": "pick_visible",
            "status": "waiting",
            "reason": "visible cube bearing too wide; rescan instead of large turn",
            "source": source_label,
            "target_color": target.color,
            "target_area": target.blob_area,
            "target_area_ratio": _area_ratio(target),
            "target_bearing": target_bearing,
            "needs_rules": True,
        }

    if not _is_visible_pick_candidate(target):
        return {
            "action": "pick_visible",
            "status": "waiting",
            "reason": "visible cube candidate too small; avoid chasing shelf/floor blob",
            "source": source_label,
            "target_color": target.color,
            "target_area": target.blob_area,
            "target_area_ratio": _area_ratio(target),
            "target_bearing": target_bearing,
            "needs_rules": True,
        }

    await safe_turn_to_bearing(
        ctx,
        target_bearing,
        tolerance_deg=12.0,
        max_turn_deg=PICK_VISIBLE_MAX_BEARING_DEG,
    )

    auto_pick = await _close_in_and_pick_color(ctx, target_color=target.color, pick_threshold=5000)
    await _sync_held_from_scene(ctx, memory)

    if memory.held_color in TARGET_COLORS:
        memory.failed_attempts["pick_missed"] = 0
        if source_label in {"front_visible"} or source_label.startswith("source_a") or _looks_like_conveyor_source(source_detections):
            try:
                status = await get_robot_status(ctx)
                _A_POSITION = _robot_xy(status)
                memory.recent_a_position = _A_POSITION
            except Exception:
                pass
        return {
            "action": "pick_visible",
            "status": "ok",
            "held_color": memory.held_color,
            "source": source_label,
            "auto_pick": auto_pick,
            "target_color": target.color,
            "target_area": target.blob_area,
            "target_area_ratio": _area_ratio(target),
            "target_bearing": target_bearing,
        }

    memory.failed_attempts["pick_missed"] = memory.failed_attempts.get("pick_missed", 0) + 1
    if memory.failed_attempts["pick_missed"] >= 2:
        if source_label.startswith("source_a"):
            _A_POSITION = None
        memory.active_color = None
        return {
            "action": "pick_visible",
            "status": "failed",
            "reason": "pick missed repeatedly; reset source search",
            "source": source_label,
            "target_color": target.color,
            "target_area": target.blob_area,
            "target_area_ratio": _area_ratio(target),
            "target_bearing": target_bearing,
            "needs_rules": True,
        }

    return {
        "action": "pick_visible",
        "status": "waiting",
        "reason": "pick missed or no target cube in range",
        "source": source_label,
        "next_target": memory.active_color,
        "target_color": target.color,
        "target_area": target.blob_area,
        "target_area_ratio": _area_ratio(target),
        "target_bearing": target_bearing,
    }


async def deterministic_pick_visible_or_a(ctx: Any, memory: AgentMemory) -> dict[str, Any]:
    """눈앞의 가까운 큐브를 먼저 집고, 없으면 A source를 찾는다."""
    dets, turned_back = await scan_for_pick_cubes(ctx)
    target_cubes = _visible_pick_candidates(dets)
    _preview_next_target(dets, memory)
    if target_cubes:
        return await _pick_from_candidates(
            ctx,
            memory,
            target_cubes,
            dets,
            source_label="front_visible",
        )

    if _A_POSITION is None:
        _log_jsonl({
            "event": "pick_source_gate",
            "status": "seek_a",
            "reason": "a_position_missing",
        })
        return await deterministic_pick_at_a(ctx, memory)

    if not await _near_recorded_a(ctx):
        _log_jsonl({
            "event": "pick_source_gate",
            "status": "return_to_a",
            "reason": "not_near_recorded_a",
            "a_position": [round(_A_POSITION[0], 3), round(_A_POSITION[1], 3)],
        })
        return await deterministic_pick_at_a(ctx, memory)

    return await deterministic_pick_at_a(ctx, memory)


async def _close_in_and_blind_pick(
    ctx: Any, memory: AgentMemory, *, source_label: str
) -> dict[str, Any] | None:
    """source 군집 blob의 bearing/distance로 접근 xy를 추정해 go_to한 뒤 nearest cube blind pick.

    인접 큐브가 합쳐진 큰 blob은 _is_cube_like를 통과 못 하므로, 개별 큐브를 분리하지 않고
    source 군집 대표 blob에 수학 기반 좌표 추정으로 다가간 뒤 pick_entity로 가까운 큐브를 잡는다.
    """
    try:
        status = await get_robot_status(ctx)
    except Exception:
        status = None
    try:
        dets = await scan_head(ctx, yaws=(-0.5, 0.0, 0.5), pitch=0.15)
    except Exception:
        dets = []
    blobs = [
        d for d in dets
        if getattr(d, "color", None) in TARGET_COLORS
        and abs(float(getattr(d, "full_bearing_deg", getattr(d, "angle_deg", 0.0)) or 0.0)) <= 55.0
        and _centroid_y_ratio(d) >= 0.22
    ]

    # source 대표 blob으로 approach xy 추정 후 go_to (수학 기반 navigation).
    if status is not None and blobs:
        target = max(blobs, key=lambda d: int(getattr(d, "blob_area", 0) or 0))
        raw_dist = _distance_from_blob_area(int(getattr(target, "blob_area", 0) or 0))
        dist = min(max(raw_dist, 0.6), 3.5)
        try:
            src_xy = _world_xy_from_detection(status, target, dist)
            rx, ry = _robot_xy(status)
            ux = src_xy[0] - rx
            uy = src_xy[1] - ry
            norm = math.hypot(ux, uy)
            if norm > 1e-6:
                ux, uy = ux / norm, uy / norm
                approach = (src_xy[0] - 0.9 * ux, src_xy[1] - 0.9 * uy)
                if _point_in_safe_world(approach[0], approach[1]) and not _point_in_no_go(approach[0], approach[1]):
                    _log_jsonl({"event": "blind_pick_approach", "source": source_label,
                                "approach_xy": [round(approach[0], 3), round(approach[1], 3)],
                                "est_dist": round(dist, 2)})
                    try:
                        await _go_to_xy(ctx, approach[0], approach[1])
                    except Exception as exc:
                        _log_jsonl({"event": "blind_pick_approach_failed", "error": str(exc)})
            bearing = float(getattr(target, "full_bearing_deg", getattr(target, "angle_deg", 0.0)) or 0.0)
            if abs(bearing) > 8.0:
                await safe_turn_to_bearing(ctx, bearing, tolerance_deg=12.0, max_turn_deg=45.0)
        except Exception as exc:
            _log_jsonl({"event": "blind_pick_approach_error", "error_type": type(exc).__name__, "error": str(exc)})

    # 1~2회 전진 + blind pick. source가 더 안 보이면 전진 중단(지나칠 위험).
    for step in range(2):
        pick = await _try_pick_nearest_at_source(ctx, memory, source_label=source_label)
        if pick is not None:
            return pick
        try:
            dets2 = await scan_head(ctx, yaws=(-0.4, 0.0, 0.4), pitch=0.15)
        except Exception:
            dets2 = []
        if not [d for d in dets2 if getattr(d, "color", None) in TARGET_COLORS and _centroid_y_ratio(d) >= 0.22]:
            break
        await move_velocity(ctx, vx=0.16, duration_s=1.0)
    return None


async def deterministic_pick_at_a(ctx: Any, memory: AgentMemory) -> dict[str, Any]:
    """A로 이동 → source 군집으로 close-in → blind pick(1차) → 실패 시 visible 후보 pick.

    인접 큐브가 합쳐 보이는 A source에서는 개별 큐브 분리 대신 blind pick을 1차 전략으로 쓴다.
    """
    global _A_POSITION
    if not await _ensure_at_conveyor_a(ctx, memory):
        return {
            "action": "pick_at_a",
            "status": "failed",
            "reason": "conveyor A sign not found",
            "needs_rules": True,
        }

    # 1차: source 군집으로 close-in 후 blind pick.
    blind = await _close_in_and_blind_pick(ctx, memory, source_label="source_a_blind")
    if blind is not None:
        blind["action"] = "pick_at_a"
        blind["next_after_deliver"] = memory.active_color
        return blind

    # 2차: visible 후보가 잡히면 후보 pick.
    dets = await scan_head(ctx, yaws=(-0.6, 0.0, 0.6), pitch=0.15)
    target_cubes = _visible_pick_candidates(dets)
    _preview_next_target(dets, memory)
    if target_cubes:
        result = await _pick_from_candidates(ctx, memory, target_cubes, dets, source_label="source_a")
        result["action"] = "pick_at_a"
        result["next_after_deliver"] = memory.active_color
        return result

    # 3차: 소폭 회전 후 close-in blind pick 재시도.
    memory.failed_attempts["a_no_cube"] = memory.failed_attempts.get("a_no_cube", 0) + 1
    try:
        await set_head(ctx, yaw=0.0, pitch=0.15)
    except Exception:
        pass
    await move_velocity(ctx, wz=0.35, duration_s=0.8)
    await asyncio.sleep(0.2)
    blind2 = await _close_in_and_blind_pick(ctx, memory, source_label="source_a_blind_turn")
    if blind2 is not None:
        blind2["action"] = "pick_at_a"
        blind2["next_after_deliver"] = memory.active_color
        return blind2

    if memory.failed_attempts["a_no_cube"] >= 2:
        _A_POSITION = None
        memory.active_color = None
        return {
            "action": "pick_at_a",
            "status": "failed",
            "reason": "no target cube at recorded A; reset A search",
            "needs_rules": True,
        }
    return {
        "action": "pick_at_a",
        "status": "waiting",
        "reason": "no target cube at A",
        "next_target": memory.active_color,
    }


async def deterministic_deliver_to_pad(ctx: Any, memory: AgentMemory) -> dict[str, Any]:
    """recorded pad go_to(대략 접근) → visual creep(정밀 접근) → place → A 복귀.

    One-shot 거리 공식만으로는 원거리에서 오차가 커서 패드를 벗어나므로,
    pick과 동일하게 재스캔+전진을 반복하는 closed-loop 접근으로 실제 패드 앞까지 다가간다."""
    held = memory.held_color
    if not held:
        return {"action": "deliver", "status": "failed", "reason": "not holding"}
    if held not in TARGET_COLORS:
        return {"action": "deliver", "status": "failed", "reason": f"unknown color {held}"}
    if held == "green" and not _is_pad_sign_confirmed(held):
        _invalidate_pad_memory(held, "green_delivery_requires_c_sign")

    def _finish_ok(xy: tuple[float, float], auto_place: Any) -> dict[str, Any]:
        placed_color = held
        _store_pad_memory(placed_color, success_xy=xy, approach_xy=xy)
        if placed_color not in memory.completed_colors:
            memory.completed_colors.append(placed_color)
        memory.stage = "need_cube"
        _sync_memory_views(memory)
        return {
            "action": "deliver",
            "status": "ok",
            "placed_color": placed_color,
            "auto_place": auto_place,
            "approach_pose_source": "success_pose",
        }

    # 1) 대략 접근: success pose 우선, 없으면 pad center + 현재 robot pose로 reflection 재계산.
    status = await get_robot_status(ctx)
    robot_xy = _robot_xy(status)
    approach_xy, approach_source = _resolve_deliver_approach(held, robot_xy)
    pad_center = _valid_recorded_pad(held)
    if approach_xy is not None:
        _store_pad_memory(held, center_xy=pad_center, approach_xy=approach_xy)

    if approach_xy and not _too_close_to_a(approach_xy[0], approach_xy[1]):
        _log_jsonl({"event": "deliver_step", "step": "go_to", "held_color": held,
                    "xy": list(approach_xy), "status": "start",
                    "approach_pose_source": approach_source,
                    "expected_sign": DESTINATION_SIGN_RULES.get(held),
                    "reflection_side": "left"})
        try:
            await _go_to_deliver_xy(ctx, held, approach_xy[0], approach_xy[1])
            _log_jsonl({"event": "deliver_step", "step": "go_to", "held_color": held,
                        "xy": list(approach_xy), "status": "done",
                        "approach_pose_source": approach_source,
                        "expected_sign": DESTINATION_SIGN_RULES.get(held),
                        "reflection_side": "left"})
        except Exception as exc:
            print(f"  deliver rough go_to failed (continuing to visual creep): {exc}")
            _log_jsonl({"event": "deliver_step", "step": "go_to", "held_color": held,
                        "xy": list(approach_xy), "status": "failed", "error": str(exc),
                        "approach_pose_source": approach_source, "reflection_side": "left"})
    else:
        _log_jsonl({"event": "deliver_step", "step": "go_to", "held_color": held,
                    "status": "skipped", "approach_pose_source": approach_source,
                    "expected_sign": DESTINATION_SIGN_RULES.get(held),
                    "reason": "no proven pad memory; use rotation search + creep"})
        found_xy = await _search_pad_xy_by_rotation(ctx, held)
        if found_xy is not None:
            approach_xy = found_xy
            approach_source = "vision_reflection"
            try:
                await _go_to_deliver_xy(ctx, held, found_xy[0], found_xy[1])
            except Exception as exc:
                _log_jsonl({"event": "deliver_step", "step": "go_to", "held_color": held,
                            "xy": list(found_xy), "status": "failed", "error": str(exc),
                            "approach_pose_source": approach_source, "reflection_side": "left"})

    # 2) 정밀 접근: 스캔+회전+전진 반복으로 실제 패드 blob이 충분히 커질 때까지 접근
    reached = await _visual_creep_to_pad(ctx, held)
    _log_jsonl({"event": "deliver_step", "step": "creep", "held_color": held, "reached": reached})
    if not reached and approach_source in {"success_pose", "recorded_center_reflection"}:
        _invalidate_pad_memory(held, f"{approach_source}_creep_failed")
    if reached:
        trust_recorded = held != "green" or _is_pad_sign_confirmed(held)
        auto_place = await _close_in_and_place(ctx, held, trust_recorded=trust_recorded)
        _log_jsonl({"event": "deliver_step", "step": "place", "held_color": held,
                    "result": auto_place})
        await _sync_held_from_scene(ctx, memory)
        if memory.held_color is None:
            try:
                status = await get_robot_status(ctx)
                rx, ry = _robot_xy(status)
            except Exception:
                rx, ry = approach_xy or (0.0, 0.0)
            result = _finish_ok((rx, ry), auto_place)
            try:
                memory.delivered_count = await get_delivered_count(ctx)
            except Exception:
                pass
            await _return_to_a_and_preview(ctx, memory)
            return result
        _PAD_APPROACH_POSES.pop(held, None)
        print(f"  deliver failed after visual creep (reached but place failed): auto_place={auto_place}")

    # 3) 실패 시: 같은 pad center를 반대쪽 offset으로 한 번 더 접근
    center = _valid_recorded_pad(held)
    if center is not None:
        try:
            status = await get_robot_status(ctx)
            retry_xy = _safe_pad_approach_pose(_robot_xy(status), center, retry=True)
            if not _too_close_to_a(retry_xy[0], retry_xy[1], margin=1.1):
                _store_pad_memory(held, center_xy=center, approach_xy=retry_xy)
                _log_jsonl({"event": "deliver_step", "step": "go_to_retry", "held_color": held,
                            "xy": list(retry_xy), "reflection_side": "right"})
                await _go_to_deliver_xy(ctx, held, retry_xy[0], retry_xy[1])
                reached = await _visual_creep_to_pad(ctx, held, max_steps=5)
                await safe_lateral_offset(ctx, left=False, duration_s=0.35, vy=0.12)
                trust_recorded = held != "green" or _is_pad_sign_confirmed(held)
                auto_place = await _close_in_and_place(ctx, held, trust_recorded=trust_recorded) if reached else None
                await _sync_held_from_scene(ctx, memory)
                if memory.held_color is None:
                    status = await get_robot_status(ctx)
                    result = _finish_ok(_robot_xy(status), auto_place)
                    try:
                        memory.delivered_count = await get_delivered_count(ctx)
                    except Exception:
                        pass
                    await _return_to_a_and_preview(ctx, memory)
                    return result
        except Exception as exc:
            _log_jsonl({"event": "deliver_step", "step": "go_to_retry", "held_color": held,
                        "status": "failed", "error": str(exc), "reflection_side": "right"})

    # 4) 패드가 전혀 안 보이면 회전 스캔으로 다시 찾고 재시도 1회
    _PAD_APPROACH_POSES.pop(held, None)
    pad_xy = await _search_pad_xy_by_rotation(ctx, held)
    if pad_xy is None:
        return {"action": "deliver", "status": "failed", "reason": "pad not found after rotation", "needs_rules": True}
    try:
        await _go_to_xy(ctx, pad_xy[0], pad_xy[1])
    except Exception as exc:
        return {"action": "deliver", "status": "failed", "reason": f"go_to visual error: {exc}", "needs_rules": True}
    reached = await _visual_creep_to_pad(ctx, held)
    trust_recorded = held != "green" or _is_pad_sign_confirmed(held)
    auto_place = await _close_in_and_place(ctx, held, trust_recorded=trust_recorded) if reached else None
    await _sync_held_from_scene(ctx, memory)
    if memory.held_color is None:
        try:
            status = await get_robot_status(ctx)
            rx, ry = _robot_xy(status)
        except Exception:
            rx, ry = pad_xy
        result = _finish_ok((rx, ry), auto_place)
        try:
            memory.delivered_count = await get_delivered_count(ctx)
        except Exception:
            pass
        await _return_to_a_and_preview(ctx, memory)
        return result

    print(f"  deliver visual failed: reached={reached} auto_place={auto_place}")
    return {"action": "deliver", "status": "failed",
            "reason": {"reached": reached, "auto_place": auto_place}, "needs_rules": True}


async def execute_decision(
    ctx: Any,
    decision: AgentDecision,
    observation: Observation,
    memory: AgentMemory,
) -> dict[str, Any]:
    """검증된 LLM 결정 하나를 Level 1 robot 행동으로 변환합니다.

    search: 회전하며 재스캔, 큐브가 없으면 A 컨베이어 방향으로 이동.
    navigate: bearing+거리 추정 xy로 go_to_xy (패드는 기록 좌표 우선).
    pick/place: 근처 도착 후 SDK 호출. recover/skip/stop 정책 포함.
    """
    action = decision.next_action
    target_color = decision.target_color or memory.active_color

    # 전도 가드: 넘어진 상태면 navigate/pick/place 시도 않고 대기(전도 복구 액션 없음).
    if _robot_is_fallen(observation.robot_status) and action not in {"recover", "stop"}:
        return {"action": "noop_fallen", "status": "robot_fallen_wait", "intended": action}

    if action in {"search_cube", "search_pad"}:
        # 큐브 탐색: 큐브-like 후보가 있으면 회전 없이 다음 navigate 유도.
        # signage/pad 잡동사니만 있으면(큐브 없음) 실제로 회전/A 이동.
        cube_cands = _visible_pick_candidates(observation.detections) if action == "search_cube" else []
        if action == "search_cube" and cube_cands:
            memory.search_turns = 0
            return {"action": action, "status": "cube_visible_no_turn", "detections": len(cube_cands)}
        if action == "search_cube" and not cube_cands and memory.search_turns == 0:
            _log_jsonl({"event": "cube_scan_front_only", "reason": "search_cube_no_close_visible"})
            try:
                await set_head(ctx, yaw=0.0, pitch=0.15)
            except Exception:
                pass
            await move_velocity(ctx, wz=0.35, duration_s=0.8)
            memory.search_turns += 1
            await scan_head(ctx, yaws=SCAN_HEAD_YAWS, pitch=0.15)
            return {"action": action, "status": "small_turn_and_scanned", "turn": memory.search_turns}
        if action == "search_pad" and _filter_pads(observation.detections, memory.held_color or target_color):
            memory.search_turns = 0
            return {"action": action, "status": "pad_visible_no_turn"}
        if memory.search_turns < 4:
            await move_velocity(ctx, wz=0.35, duration_s=0.8)
            memory.search_turns += 1
            await scan_head(ctx, yaws=(-0.8, 0.0, 0.8), pitch=0.15)
            return {"action": action, "status": "turned_and_scanned", "turn": memory.search_turns}
        # 큐브 탐색 한계 → A 컨베이어로 이동
        if action == "search_cube":
            memory.search_turns = 0
            return await navigate_toward_conveyor_a(ctx, memory)
        memory.search_turns = 0
        return await recover_motion(ctx, memory, reason="search_pad exhausted")

    if action in {"navigate_to_cube", "navigate_to_pad"}:
        if action == "navigate_to_pad":
            # 패드 이동: held_color 고정, 큐브 blob으로 navigate 금지
            held = memory.held_color or target_color
            if not held:
                return {"action": action, "status": "failed", "reason": "no held_color for pad navigation"}
            target_color = held
            recorded = _valid_recorded_approach(held) or _valid_recorded_pad(held)
            if recorded:
                px, py = recorded
                try:
                    res = await _go_to_deliver_xy(ctx, held, px, py)
                except Exception as exc:
                    return {"action": action, "status": "go_to_failed", "error": str(exc)}
                memory.search_turns = 0
                res_summary = result_summary(res)
                auto_place = None
                if res_summary.get("status") == "done":
                    trust_recorded = held != "green" or _is_pad_sign_confirmed(held)
                    auto_place = await _close_in_and_place(ctx, held, trust_recorded=trust_recorded)
                return {"action": action, "target_color": held, "target_xy": [px, py],
                        "used_recorded_pad": True, "approach_pose_source": "recorded",
                        "expected_sign": DESTINATION_SIGN_RULES.get(held),
                        "result": res_summary, "auto_place": auto_place}
            estimated = estimate_pad_approach_from_observation(observation, held)
            if estimated is None:
                return {"action": action, "status": "failed",
                        "reason": f"no {DESTINATION_SIGN_RULES.get(held)} pad sign visible; need search_pad"}
            pad_xy, pad_center, pad_det = estimated
            if held == "green" and not _is_pad_sign_confirmed(held):
                if not await _confirm_destination_sign_quick(ctx, held, yaws=(0.0,)):
                    _invalidate_pad_memory(held, "green_navigate_without_c_sign")
                    return {"action": action, "status": "failed",
                            "reason": "green C sign not confirmed; need search_pad"}
            _store_pad_memory(
                held,
                center_xy=pad_center,
                approach_xy=pad_xy,
                sign_bearing=float(getattr(pad_det, "full_bearing_deg", getattr(pad_det, "angle_deg", 0.0)) or 0.0),
            )
            try:
                res = await _go_to_deliver_xy(ctx, held, pad_xy[0], pad_xy[1])
            except Exception as exc:
                return {"action": action, "status": "go_to_failed", "error": str(exc)}
            memory.search_turns = 0
            res_summary = result_summary(res)
            auto_place = None
            if res_summary.get("status") == "done":
                trust_recorded = held != "green" or _is_pad_sign_confirmed(held)
                auto_place = await _close_in_and_place(ctx, held, trust_recorded=trust_recorded)
            return {"action": action, "target_color": held, "target_xy": list(pad_xy),
                    "pad_center_xy": list(pad_center), "approach_pose_source": "vision_reflection",
                    "expected_sign": DESTINATION_SIGN_RULES.get(held),
                    "result": res_summary, "auto_place": auto_place}

        # navigate_to_cube — 가장 가까운(중앙·큰) visible cube로 이동 후 pick
        cubes = _candidate_pick_cubes(observation.detections)
        if cubes:
            target = _pick_preferred_cube(cubes) or cubes[0]
            target_color = target.color
            memory.active_color = target.color
        target_xy = estimate_target_xy_from_observation(observation, target_color)
        if target_xy is None:
            return {"action": action, "status": "failed", "reason": "no coordinate estimate"}
        try:
            res = await _go_to_xy(ctx, *target_xy)
        except Exception as exc:
            return {"action": action, "status": "go_to_failed", "error": str(exc)}
        memory.search_turns = 0
        if action == "navigate_to_cube":
            try:
                auto_pick = await _close_in_and_pick(ctx)
            except Exception as exc:
                return {"action": action, "target_color": target_color, "target_xy": list(target_xy),
                        "result": result_summary(res), "auto_pick_error": str(exc)}
            return {"action": action, "target_color": target_color, "target_xy": list(target_xy),
                    "result": result_summary(res), "auto_pick": auto_pick}

    if action == "pick_cube":
        # 전도 방지 가드: 정면에 잡을 큐브(area>=6000)가 보일 때만 pick. 안 보이면 back-off.
        try:
            await set_head(ctx, yaw=0.0, pitch=0.15)
        except Exception:
            pass
        dets = await scan_head(ctx, yaws=(0.0,), pitch=0.15)
        cubes = [c for c in _filter_cubes(dets) if c.color in TARGET_COLORS]
        if not target_color and cubes:
            target = _pick_preferred_cube(cubes) or cubes[0]
            target_color = target.color
        if target_color and target_color not in TARGET_COLORS:
            return {"action": "pick_cube", "target_color": target_color,
                    "status": "skipped_non_target", "reason": "unknown target color"}
        if not cubes or cubes[0].blob_area < 6000:
            await move_velocity(ctx, vx=-0.15, duration_s=0.6)
            return {"action": "pick_cube", "target_color": target_color,
                    "status": "skipped_no_close_cube", "nearest_area": cubes[0].blob_area if cubes else 0}
        res = await pick_nearest_cube(ctx)
        return {"action": "pick_cube", "target_color": target_color, "result": result_summary(res)}

    if action == "place_cube":
        held = memory.held_color
        if not held:
            return {"action": "place_cube", "status": "failed", "reason": "not holding"}
        auto_place = await _close_in_and_place(ctx, held)
        if auto_place:
            return {"action": "place_cube", "target_color": held, "result": auto_place}
        return {"action": "place_cube", "target_color": held,
                "status": "skipped_not_at_destination",
                "expected_sign": DESTINATION_SIGN_RULES.get(held)}

    if action == "recover":
        return await recover_motion(ctx, memory, reason=decision.reason)

    if action == "skip_target":
        color = target_color or memory.active_color or memory.held_color
        if color and color not in memory.skipped_colors:
            memory.skipped_colors.append(color)
        memory.held_color = None
        memory.active_color = None
        memory.stage = "need_cube"
        return {"action": "skip_target", "color": color}

    return {"action": action, "status": "no_op"}


async def run_agent(
    ctx: Any,
    *,
    max_cycles: int = 10_000,
    completion: CompletionConfig | None = None,
    calibrate: bool = False,
    calibrate_budget_s: float = 45.0,
) -> AgentMemory:
    """Level 1 의사결정 루프: 텍스트 LLM이 매 cycle 상위 행동을 선택하고 결정론적 실행 함수가 저수준을 수행."""
    global _run_start_ts
    memory = AgentMemory()
    last_result: dict[str, Any] | None = None
    tracker = CompletionTracker(completion) if completion is not None else None
    consecutive_failures = 0
    _run_start_ts = time.time()
    prev_run_delivered = 0

    async def _timed(awaitable: Any, label: str) -> Any:
        if tracker is None:
            return await awaitable
        return await tracker.wait_for_remaining(awaitable, label)

    if calibrate:
        print("\n[Level 1] Pad calibration sweep (before timer)...")
        try:
            await calibrate_pads(ctx, budget_s=calibrate_budget_s)
        except Exception as exc:
            print(f"[calibrate] error (continuing): {exc}")

    for cycle in range(1, max_cycles + 1):
        if tracker is not None:
            first_cycle = tracker.started_at is None
            tracker.start_first_cycle()
            if first_cycle:
                tracker.print_start()
            if tracker.started_at is not None:
                _elapsed = tracker.elapsed_s()
                limit = completion.max_elapsed_s if completion else 600.0
                if _elapsed >= limit:
                    tracker.mark_ended(f"elapsed {_elapsed:.1f}/{limit:.1f} seconds (wall-clock guard)")
                    print(f"Completion target reached before cycle action: elapsed {_elapsed:.1f}s (wall-clock guard).")
                    break
            try:
                reason = await tracker.stop_reason_from_scene(ctx)
            except (TimeoutError, OSError, Exception):
                reason = None
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached before cycle action: {reason}.")
                break

        try:
            cycle_t0 = time.perf_counter()
            await _sync_held_from_scene(ctx, memory)
            try:
                memory.delivered_count = await get_delivered_count(ctx)
            except Exception:
                pass
            rd = _run_delivered(memory.delivered_count)
            _sync_memory_views(memory)
            memory.recent_a_position = _A_POSITION
            if cycle == 1:
                _log_score(memory.delivered_count)
            if rd != prev_run_delivered:
                _log_throughput(memory.delivered_count)
                _log_score(memory.delivered_count)
                prev_run_delivered = rd

            elapsed_run = round(time.time() - _run_start_ts, 1) if _run_start_ts else None
            score = _score(rd)
            print(f"\n[Level 1] Cycle {cycle} stage={memory.stage} held={memory.held_color} "
                  f"delivered={memory.delivered_count} run={rd} score={score} "
                  f"pads={list(_PAD_POSITIONS.keys())}")

            try:
                st = await get_robot_status(ctx)
                rx, ry = _robot_xy(st)
                robot_xy = [rx, ry]
            except Exception:
                st = None
                robot_xy = None
            _log_jsonl({
                "event": "cycle_start",
                "cycle": cycle,
                "stage": memory.stage,
                "held_color": memory.held_color,
                "delivered_count": memory.delivered_count,
                "run_delivered": rd,
                "score": score,
                "elapsed_s": elapsed_run,
                "robot_xy": robot_xy,
                "pad_positions": dict(_PAD_POSITIONS),
                "pad_approach_poses": dict(_PAD_APPROACH_POSES),
                "place_success_poses": dict(_PLACE_SUCCESS_POSES),
                "last_sign_bearing": dict(_LAST_SIGN_BEARING),
                "consecutive_failures": consecutive_failures,
                "decision_source": memory.decision_source,
            })
            # === 상위 의사결정: 텍스트 LLM이 매 cycle 다음 행동을 선택한다 ===
            # 구조: LLM은 고수준 행동 카테고리(pick/deliver/recover/skip/stop)와 target_color를 결정하고,
            # 결정론적 실행 함수가 저수준 이동·pick·place를 수행한다.
            # (고수준 추론 = LLM, 저수준 제어 = 결정론 — 실시간 로봇 제어의 표준 분업.)
            observation = await _timed(observe_world(ctx, memory, use_vlm=False), "observe_world")
            try:
                decision = await _timed(decide_next_action(TASK, observation, memory, last_result), "decide_next_action")
            except Exception as exc:
                decision = AgentDecision(next_action="recover",
                                         reason=f"llm_exception: {exc}", recovery_strategy="llm_error")
            decision_source = "llm"
            memory.decision_source = decision_source
            memory.last_llm_decision = decision.__dict__
            _log_jsonl({"event": "llm_decision", "cycle": cycle,
                        "next_action": decision.next_action, "target_color": decision.target_color,
                        "reason": decision.reason, "recovery_strategy": decision.recovery_strategy})

            # LLM 호출 자체가 실패한 경우에만 규칙 fallback으로 보정 (LLM 우회가 아닌 장애 시 안전망).
            if decision.recovery_strategy in ("llm_error", "bad_llm_output"):
                decision_source = "rules_fallback"
                memory.decision_source = decision_source
                decision = decide_next_action_rules(observation, memory, last_result)
                _log_jsonl({"event": "rules_fallback", "cycle": cycle,
                            "next_action": decision.next_action, "reason": decision.reason})

            if decision.next_action == "stop":
                memory.logs.append({"event": "stop", "reason": decision.reason, "cycle": cycle})
                _log_jsonl({"event": "stop", "cycle": cycle, "reason": decision.reason})
                print(f"  LLM stop: {decision.reason}")
                break

            action_result: dict[str, Any]
            pad_actions = {"search_pad", "navigate_to_pad", "place_cube", "deliver"}
            cube_actions = {"search_cube", "navigate_to_cube", "pick_cube"}

            if decision.next_action == "skip_target":
                skip_color = decision.target_color or memory.active_color or memory.held_color
                if skip_color and skip_color not in memory.skipped_colors:
                    memory.skipped_colors.append(skip_color)
                memory.held_color = None
                memory.active_color = None
                memory.stage = "need_cube"
                action_result = {"action": "skip_target", "status": "ok", "color": skip_color}
            elif decision.next_action == "recover":
                action_result = await _timed(recover_motion(ctx, memory, reason=decision.reason), "recover_motion")
            elif memory.held_color in TARGET_COLORS and decision.next_action in pad_actions:
                action_result = await _timed(deterministic_deliver_to_pad(ctx, memory), "deterministic_deliver_to_pad")
            elif memory.held_color in TARGET_COLORS:
                # LLM이 큐브를 들고 있는데 cube 계열을 골랐을 때의 이중 안전 보정 → deliver로.
                action_result = await _timed(deterministic_deliver_to_pad(ctx, memory), "deterministic_deliver_to_pad")
            elif decision.next_action in cube_actions or memory.held_color is None:
                action_result = await _timed(deterministic_pick_visible_or_a(ctx, memory), "deterministic_pick_visible_or_a")
            else:
                action_result = await _timed(deterministic_pick_visible_or_a(ctx, memory), "deterministic_pick_visible_or_a")

            last_result = action_result
            if action_result.get("status") == "ok":
                consecutive_failures = 0
                print(f"  LLM OK [{decision.next_action}]: {action_result.get('action')} "
                      f"held={memory.held_color} placed={action_result.get('placed_color')} "
                      f"next={memory.active_color}")
                _log_jsonl({"event": "llm_executed", "cycle": cycle, "status": "ok",
                            "decision_source": decision_source, "next_action": decision.next_action,
                            "target_color": decision.target_color, **action_result})
            elif action_result.get("status") == "waiting":
                print(f"  LLM waiting [{decision.next_action}] (next={memory.active_color}): {action_result.get('reason')}")
                _log_jsonl({"event": "llm_executed", "cycle": cycle, "status": "waiting",
                            "decision_source": decision_source, "next_action": decision.next_action, **action_result})
            else:
                consecutive_failures += 1
                print(f"  LLM action failed [{decision.next_action}] (failures={consecutive_failures}): {action_result}")
                _log_jsonl({"event": "llm_executed", "cycle": cycle, "status": "failed",
                            "decision_source": decision_source, "next_action": decision.next_action,
                            "consecutive_failures": consecutive_failures, **action_result})
                # 연속 실패 시에도 LLM이 다음 cycle에서 recover/skip/stop을 스스로 선택하도록 맡긴다.


            _log_jsonl({
                "event": "cycle_time",
                "cycle": cycle,
                "seconds": round(time.perf_counter() - cycle_t0, 3),
                "decision_source": decision_source,
            })

        except CompletionTimeout as exc:
            print(f"[cycle {cycle}] completion timeout: {exc}")
            if tracker is not None:
                tracker.mark_ended(str(exc))
            _log_jsonl({"event": "completion_timeout", "cycle": cycle, "error": str(exc)})
            break
        except Exception as exc:
            print(f"[cycle {cycle}] error isolated: {type(exc).__name__}: {exc}")
            memory.logs.append({"event": "cycle_error", "cycle": cycle, "error": str(exc)})
            _log_jsonl({"event": "cycle_error", "cycle": cycle, "error": str(exc),
                          "error_type": type(exc).__name__})
            consecutive_failures += 1
            await asyncio.sleep(2.0)
            continue

        if tracker is not None:
            try:
                reason = await tracker.stop_reason_from_scene(ctx)
            except (TimeoutError, OSError, Exception):
                reason = None
            if reason is not None:
                tracker.mark_ended(reason)
                print(f"Completion target reached after cycle action: {reason}.")
                break

    if tracker is not None:
        try:
            await tracker.print_summary_from_scene(ctx)
        except Exception as exc:
            print(f"[summary] skipped (sim unreachable): {type(exc).__name__}: {exc}")
    return memory


async def run(ctx: Any) -> None:
    print(TASK)
    print("Level 1 adaptive-navigation project starter 실행 (10분 scored sim)")
    memory = await run_agent(
        ctx,
        max_cycles=10_000,
        completion=completion_config_for_round(
            level=1,
            round_name="round2",
            max_delivered_cubes=DEFAULT_MAX_DELIVERED_CUBES,
        ),
        calibrate=False,
    )
    print("\n실행 완료.")
    print(f"Delivered count: {memory.delivered_count}")
    print(f"Completed colors: {memory.completed_colors}")
    print(f"Skipped colors: {memory.skipped_colors}")
    print(f"Pad positions recorded: {dict(_PAD_POSITIONS)}")


## Connect Robot

Open the viewer URL in Chrome when prompted.


In [ ]:
from menlo_runner.config import load_config
from menlo_runner.context import open_robot_context

config = load_config(require_tokamak=True)
ctx = await open_robot_context(config, name_prefix="level-1-notebook")
print(ctx.viewer_url)


## Run Agent

Run after completing enough TODOs for your current experiment.


In [ ]:
memory = await run_agent(
    ctx,
    max_cycles=10_000,
    completion=CompletionConfig(level=1, max_elapsed_s=600),
)
memory.logs


In [ ]:
# Cleanup when finished.
# await ctx.close()
